# Distributed Multidisciplinary Design Optimization Benchmarking

<img src="https://raw.githubusercontent.com/Ahmed-Bayoumy/DMDO/DEV/DMDO_logo.png" alt="DMDO Logo" width="300"/>

<p align="justify"> In this report, the Distributed Multidisciplinary Design Optimization (DMDO) python tool is benchmarked on Supersonic Business Jet problem which is considered as a complex engineering systems due to its high-dimensionality, large number of nonlinear constraints, and interdisciplinary couplings. For more information on DMDO and implemented methods visit <a href="https://ahmed-bayoumy.github.io/DMDO/background.html">DMDO documentation</a>.

# Distributed MDO of Supersonic Business Jet (SBJ)

<p align="justify"> The multidisciplinary design optimization of supersonic business jet (SBJ) requires balancing multiple competing objectives and constraints. In this report, the Non-Hierarchical Analytical Target Cascading (NHATC) formulation is applied to systematically coordinate the various subsystems involved in aircraft design [1]. The primary objective is to minimize the aircraft's weight while ensuring that requirements across four key disciplines—structures, aerodynamics, propulsion, and overall aircraft performance—are satisfied. The problem setup features a fixed cruise altitude of 55,000 feet and a cruise velocity of Mach 1.4, and involves 39 design variables and 46 constraints. Shared and coupling variables facilitate the integration of subsystem analyses, making this example a comprehensive validation of DMDO and demonstration on distributed multidisciplinary design optimization techniques [1].

A supersonic business jet (SBJ) design problem [1] ([described here](https://asmedigitalcollection.asme.org/mechanicaldesign/article/132/5/051002/429527/A-Nonhierarchical-Formulation-of-Analytical-Target?guestAccessKey=)) is solved using Non-Hierarchical Analytical Target Cascade (NHATC) formulation implemented in the `DMDO` Python package.

<center>

| <p style="text-align:center;"><img src="https://ahmed-bayoumy.github.io/DMDO/_images/realistic_MDO.png" alt="wing" title="SBJ-MDA" width="750p" align="center"/></p> |
|:--:|
| Fig.1 Distributed MDA of the SBJ|
</center>

<center>

| <p style="text-align:center;"><img src="https://raw.githubusercontent.com/Ahmed-Bayoumy/MDO_McGill/DEV/Figures/Functional%20dependencies.jpeg" alt="wing" title="SBJ-Fun-MDA" width="750p" align="center"/></p> |
|:--:|
| Fig.2 Distributed Functional Dependencies of the SBJ [1]|
</center>


<p align="justify"> The SBJ design problem involves four disciplines; aircraft performance, propulsion, aerodynamics, and structural. Each disciplinary analysis is part of its corresponding optimization subproblem.


The Python implementation of the disciplinary analyses are outlined in Table.1. Variables' name, lower bound, and upper bound are described in Table.2.


|Discipline                 | Subproblem  | Inputs                                                                                                                                                          | Output                                                            |  [`Disciplinary Analysis Function`](#disciplinary-analysis)       |
|---------------------------|-------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------|-------------------------------------------------------------------|---------------------------------------------------------------|
|Aircraft                   | 1           | $\text{SFC}$, $W_e$, $L/D$ $W_s$, $W_f$, $h$, $M$,                                                                                                              | $W_t$, $\text{range}$     		                                | [`SBJ_Aircraft`](#aircraft-performance)                                                      |
|Propulsion 	            | 2           | $T$, $D$, $h$, $M$,                                                                                                                                             | $\text{SFC}$, $W_e$, $\text{ESF}$, $T_{E}$, $\text{Throttle}$     | [`SBJ_propulsion`](#propulsion)                                                      |
|Aerodynamics 	            | 3           | $W_t$, $\text{ESF}$, $\theta$, $t/c$, $AR_w$, $\Lambda_w$, $S_\text{ref}$, $S_\text{ht}$, $AR_\text{ht}$, $\Lambda_\text{ht}$, $L_w$, $L_\text{ht}$, $h$, $M$   | $L$, $D$, $L/D$, $P_g$, $CLo_1$, $CLo_2$                          |  [`SBJ_aerodynamics`](#aerodynamics)                                                      |
|Structures 		        | 4           | $L$, $t/c$, $AR_w$, $\Lambda_w$, $S_\text{ref}$, $S_\text{ht}$, $AR_\text{ht}$, $\lambda$, $\mathbf{t}$, $\mathbf{t}_s$, $h$, $M$,                              | $W_s$, $W_f$, $\theta$, $\mathbf{g}_1$                            |  [`SBJ_structures`](#structures)                                                      |

<center>Table.1 Definition of disciplinary analysis files</center>



# MDO Problem Setup

The following list outlines the steps required to setup the distributed SBJ problem and to solve it using DMDO:



*   Install DMDO
*   Define the variables pair using the `YAML` input file
*   Implement disciplinary analysis Python functions
*   Implement the optimization Python functions
*   Run the instantiated MDO class




#### DMDO installation

DMDO 2604.0 requires installing the following optimization and sampling open source libraries:

* Optimization using Mesh Adaptive Direct Search (<a href="https://github.com/Ahmed-Bayoumy/OMADS">`OMADS>=2604.0`</a>)
* Samplers Library (<a href="https://github.com/Ahmed-Bayoumy/samplersLib">`samplersLib>=2604.0`</a>)

Those libraries will be automatically installed when installing `DMDO` from the terminal by running `pip install dmdo`.

In [ ]:
# DMDO installation
!pip install dmdo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.0/125.0 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.3/227.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━

#### Variables pair, Suproblems, and Coordination Setup

Linked variable pairs, independent/dependent design variables, dummy and constant parameters are defined via a `YAML` file following the compact table proposed in [2].
<center>

| Variable                	| Code designation  | Lower bound   | Upper bound       | Description                   |
|---------------------------|-------------------|---------------|-------------------|-------------------------------|
| $t/c$                   	| `tc`              | 0.01      	| 0.1     			| Thickness/chord          		|
| $AR_{\text{w}}$ 	      	| `ARw`             | 2.5      	    | 8         		| Wing aspect ratio        		|
| $\Lambda_{\text{w}}$		| `LAMBDAw`         | 40			| 70    	  		| Wing sweep angle         		|
| $S_{\text{ref}}$ 	      	| `Sref`            | 200			| 800   	  		| Wing surface area        		|
| $S_{\text{ht}}$ 			| `Sht`             | 50			| 148.9		   		| Tail surface area        		|
| $AR_{\text{ht}}$  		| `ARht`            | 2.5			| 8.5   	  		| Tail aspect ratio        		|
| $T$ 				   		| `T`               | 0.1		 	| 1.0    		    | Thrust                   		|
| $\Lambda_\text{ht}$ 	  	| `LAMBDAht`        | 40      		| 70    		    | Tail sweep               		|
| $L_{\text{w}}$ 			| `Lw`              | 0.01    		| 0.2    		    | Wing distance            		|
| $L_{\text{ht}}$ 			| `Lht`             | 1       		| 3.5    		    | Tail distance            		|
| $\mathbf{t}$ 				| `t`               | 0.1     		| 4.0    		    | Nine thicknesses         		|
| $\mathbf{t}_{\text{s}}$ 	| `ts`              | 0.1     		| 9.0    		    | Nine thicknesses         		|
| $\lambda$ 	            | `lambda`          | 0.1     		| 0.4    		    | Taper ratio              		|
| $L$ 						| `L`               | 5000    		| 100000    	  	| Total lift               		|
| $W_{\text{e}}$ 			| `We`              | 100     		| 30000  		    | Engine weight            		|
| $W_{\text{t}}$			| `Wt`              | 5000    		| 100000    	  	| Total weight             		|
| $\theta$ 		            | `theta`           | 0.2     		| 50    		    | Wing twist               		|
| ESF 		                | `ESF`             | 0.5     		| 1.5    		    | Engine scaling factor    		|
| $D$ 		                | `D`               | 1000    		| 70000	  	    	| Total drag               		|
| $W_{\text{f}}$ 		 	| `Wf`              | 5000    		| 100000    	  	| Fuel weight              		|
| $L/D$ 		         	| `LD`              | 0.1     		| 10    		    | Lift/drag ratio          		|
| SFC 		                | `SFC`             | 1       		| 4    		  	  	| Specific fuel consumption		|
| $W_{\text{s}}$ 			| `Ws`              | 5000    		| 100000    	  	| Structural weight        		|

</center>
<center>Table.2 Definition of design variables of the SBJ problem</center>

The following intermediate responses (dummy response parameters) flows from the disciplinary analyses to their corresponding subproblem but do not contribute to evaluate the subproblem design criteria:
<center>

| Variable            	| Code designation  | Lower bound   | Upper bound   | Description   			|
|-----------------------|-------------------|---------------|---------------|---------------------------|
| range 				| `range`           |      		    |          		| Aircraft range         	|
| $T_{\text{E}}$		| `Temp_E`          |      		    |          		| Engine temperature     	|
| Throttle 				| `Throttle_uA`     |      		    |          		| Engine throttle setting	|
| $P_{\text{g}}$ 		| `Pg`              |      		    |          		| Pressure gradient      	|
| $CLo_{1}$ 			| `CLo1`            |      		    |          		| Lift coefficient 1     	|
| $CLo_{2}$ 			| `CLo2`            |      		    |          		| Lift coefficient 2     	|

</center>
<center>Table.3 Definition of disciplinary dummy responses of the SBJ problem</center>

The following constant parameters are shared among all linked subproblems.
<center>

| Variable             	| Code designation  | Value   	| Description           |
|-----------------------|-------------------|-----------|-----------------------|
| $M$ 		         	| `M` 				| 1.4  		| Aircraft mach number  |
| $h$ 		         	| `h` 				| 55000		| Cruising altitude     |

</center>

<center>Table.4 Definition of constant parameters of the SBJ problem</center>

Using the disciplinary analyses in Table 1 and the variables and parameters in Tables [2-4], we can formulate the non-hierarchical analytical target cascade for the distributed MDO configuration of SBJ.

In [ ]:
#@title #####*Variables pair Definition Via Yaml Dictionary*

import yaml

import os
import warnings
import os
import shutil
# Define the YAML content as a multi-line string
yaml_content = """
# The SBJ MDO problem

variables:
# vs:  [name,     sp_index, link, coupling, lb,      bl,        ub,        dim, var_type]
  v0:  ["SFC",     1,        2,    fb,       1.0,     2.0,       4.0,       1,   "REAL"]
  v1:  ["We",      1,        2,    fb,       100.0,   15000.0,   30000.0,   1,   "REAL"]
  v2:  ["Wt",      1,        3,    ff,       5000.0,  25000.0,   100000.0,  1,   "REAL"]
  v3:  ["LD",      1,        3,    fb,       0.1,     5.00,      10.0,      1,   "REAL"]
  v4:  ["Ws",      1,        4,    fb,       5000.0,  25000.0,   100000.0,  1,   "REAL"]
  v5:  ["Wf",      1,        4,    fb,       5000.0,  25000.0,   100000.0,  1,   "REAL"]
  v6:  ["SFC",     2,        1,    ff,       1.0,     2.0,       4.0,       1,   "REAL"]
  v7:  ["We",      2,        1,    ff,       100.0,   15000.0,   30000.0,   1,   "REAL"]
  v8:  ["D",       2,        3,    fb,       1000.0,  40000.0,   70000.0,   1,   "REAL"]
  v9:  ["ESF",     2,        3,    ff,       0.5,     1.0,       1.5,       1,   "REAL"]
  v10: ["T",       2,        null, u,        0.1,     0.60,      1.0,       1,   "REAL"]
  v11: ["D",       3,        2,    ff,       1000.0,  40000.0,   70000.0,   1,   "REAL"]
  v12: ["ESF",     3,        2,    fb,       0.5,     1.0,       1.5,       1,   "REAL"]
  v13: ["Wt",      3,        1,    fb,       5000.0,  25000.0,   100000.0,  1,   "REAL"]
  v14: ["LD",      3,        1,    ff,       0.1,     5.00,      10.0,      1,   "REAL"]
  v15: ["theta",   3,        4,    fb,       0.2,     10.0,      50.0,      1,   "REAL"]
  v16: ["L",       3,        4,    ff,       5000.0,  34300.0,   100000.0,  1,   "REAL"]
  v17: ["tc",      3,        4,    s,        0.01,    0.050,     0.1,       1,   "REAL"]
  v18: ["ARw",     3,        4,    s,        2.5,     3.00,      8.0,       1,   "REAL"]
  v19: ["LAMBDAw", 3,        4,    s,        40.0,    60.0,      70.0,      1,   "REAL"]
  v20: ["Sref",    3,        4,    s,        200.0,   500.0,     800.0,     1,   "REAL"]
  v21: ["Sht",     3,        4,    s,        50.0,    100.0,     148.9,     1,   "REAL"]
  v22: ["ARht",    3,        4,    s,        2.5,     5.5,       8.5,       1,   "REAL"]
  v23: ["LAMBDAht",3,        null, u,        40.0,    45.0,      70.0,      1,   "REAL"]
  v24: ["Lw",      3,        null, u,        0.01,    0.150,     0.2,       1,   "REAL"]
  v25: ["Lht",     3,        null, u,        1.0,     1.50,      3.5,       1,   "REAL"]
  v26: ["theta",   4,        3,    ff,       0.2,     10.0,      50.0,      1,   "REAL"]
  v27: ["L",       4,        3,    fb,       5000.0,  34300.0,   100000.0,  1,   "REAL"]
  v28: ["Ws",      4,        1,    ff,       5000.0,  25000.0,   100000.0,  1,   "REAL"]
  v29: ["Wf",      4,        1,    ff,       5000.0,  25000.0,   100000.0,  1,   "REAL"]
  v30: ["tc",      4,        3,    s,        0.01,    0.064,     0.1,       1,   "REAL"]
  v31: ["ARw",     4,        3,    s,        2.5,     2.50,      8.0,       1,   "REAL"]
  v32: ["LAMBDAw", 4,        3,    s,        40.0,    70.0,      70.0,      1,   "REAL"]
  v33: ["Sref",    4,        3,    s,        200.0,   667.0,     800.0,     1,   "REAL"]
  v34: ["Sht",     4,        3,    s,        50.0,    99.7,     148.9,     1,   "REAL"]
  v35: ["ARht",    4,        3,    s,        2.5,     2.5,       8.5,       1,   "REAL"]
  v36: ["lambda",  4,        null, u,        0.1,     0.30,      0.4,       1,   "REAL"]
  v37: ["t",       4,        null, u,        0.1,     0.97,      4.0,       1,   "REAL"]
  v38: ["t",       4,        null, u,        0.1,     0.52,      4.0,       1,   "REAL"]
  v39: ["t",       4,        null, u,        0.1,     0.21,      4.0,       1,   "REAL"]
  v40: ["t",       4,        null, u,        0.1,     4.00,      4.0,       1,   "REAL"]
  v41: ["t",       4,        null, u,        0.1,     3.66,      4.0,       1,   "REAL"]
  v42: ["t",       4,        null, u,        0.1,     0.91,      4.0,       1,   "REAL"]
  v43: ["t",       4,        null, u,        0.1,     0.97,      4.0,       1,   "REAL"]
  v44: ["t",       4,        null, u,        0.1,     0.52,      4.0,       1,   "REAL"]
  v45: ["t",       4,        null, u,        0.1,     0.21,      4.0,       1,   "REAL"]
  v46: ["ts",      4,        null, u,        0.1,     2.17,      9.0,       1,   "REAL"]
  v47: ["ts",      4,        null, u,        0.1,     1.48,      9.0,       1,   "REAL"]
  v48: ["ts",      4,        null, u,        0.1,     0.76,      9.0,       1,   "REAL"]
  v49: ["ts",      4,        null, u,        0.1,     4.40,      9.0,       1,   "REAL"]
  v50: ["ts",      4,        null, u,        0.1,     4.02,      9.0,       1,   "REAL"]
  v51: ["ts",      4,        null, u,        0.1,     1.01,      9.0,       1,   "REAL"]
  v52: ["ts",      4,        null, u,        0.1,     2.17,      9.0,       1,   "REAL"]
  v53: ["ts",      4,        null, u,        0.1,     1.48,      9.0,       1,   "REAL"]
  v54: ["ts",      4,        null, u,        0.1,     0.76,      9.0,       1,   "REAL"]
  v55: ["range",   1,        null, dummy,    0.0,     0.0,       0.0,       1,   "REAL"]
  v56: ["Temp_E",  2,        null, dummy,    0.0,     0.0,       0.0,       1,   "REAL"]
  v57: ["Throttle_uA",2,     null, dummy,    0.0,     0.0,       0.0,       1,   "REAL"]
  v58: ["Pg",      3,        null, dummy,    0.0,     0.0,       0.0,       1,   "REAL"]
  v59: ["CLo1",    3,        null, dummy,    0.0,     0.0,       0.0,       1,   "REAL"]
  v60: ["CLo2",    3,        null, dummy,    0.0,     0.0,       0.0,       1,   "REAL"]

# DA1:
  #   index: 1

  #   blackbox: SBJ_A1
  #   type: callable
  #   links: [2,3,4]
  #   coupling_type: ff

  # DA2:
  #   index: 2

  #   blackbox: SBJ_A2
  #   type: callable
  #   links: [1,3]
  #   coupling_type: [ff, ff, ff]

DA:
  DA1:
    index: 1
#   inputs: ["SFC","We","LD","Ws","Wf"]
    inputs: [v0, v1, v3, v4, v5]
#   outputs: ["Wt","range"]
    outputs: [v2, v55]
    blackbox: SBJ_aircraft_analysis
    type: callable
    links: [2,3,4]
    coupling_type: ff

  DA2:
    index: 2
#   inputs: ["D","T"]
    inputs: [v8, v10]
#   outputs: ["SFC","We","ESF","Temp_E","Throttle_uA"]
    outputs: [v6,v7,v9, v56, v57]
    blackbox: SBJ_propulsion_analysis
    type: callable
    links: [1,3]
    coupling_type: ff

  DA3:
    index: 3
    # inputs: ["Wt","ESF","theta","tc","ARw","LAMBDAw","Sref","Sht","ARht","LAMBDAht","Lw","Lht"]
    inputs:   [v13,   v12,    v15, v17,  v18,      v19,   v20, v21,    v22,       v23, v24, v25 ]
    # outputs: ["L","D","LD","Pg","CLo1","CLo2"]
    outputs: [v16,v11,v14, v58, v59,v60]
    blackbox: SBJ_aerodynamics_analysis
    type: callable
    links: [1,2,4]
    coupling_type: ff

  DA4:
    index: 4
    # inputs: ["L","tc","ARw","LAMBDAw","Sref","Sht","ARht","lambda","t","ts"]
    inputs: [v27,v30,v31, v32, v33, v34, v35, v36, v37,v38,v39,v40,v41,v42,v43,v44,v45, v46, v47,v48, v49, v50,v51,v52,v53,v54]
    # outputs: ["Ws","Wf","theta"]
    outputs: [v28,v29, v26]
    blackbox: SBJ_structure_analysis
    type: callable
    links: [1,3]
    coupling_type: ff

MDA:
  MDA1:
    index: 1
    nAnalyses: 1
    analyses: [1]
    variables: [v0, v1, v3, v4, v5]
    responses:  [v2, v55]

  MDA2:
    index: 2
    nAnalyses: 1
    analyses: [2]
    variables:  [v8, v10]
    responses: [v6,v7,v9, v56, v57]

  MDA3:
    index: 3
    nAnalyses: 1
    analyses: [3]
    variables: [v13,   v12,    v15, v17,  v18,      v19,   v20, v21,    v22,       v23, v24, v25 ]
    responses:  [v16,v11,v14, v58, v59,v60]

  MDA4:
    index: 4
    nAnalyses: 1
    analyses: [4]
    variables: [v27,v30,v31, v32, v33, v34, v35, v36, v37,v38,v39,v40,v41,v42,v43,v44,v45, v46, v47,v48, v49, v50,v51,v52,v53,v54]
    responses: [v28,v29, v26]


coord:
  c1:
    index: 1
    type: ADMM
    beta: 1.8
    gamma: 0.9
    nsp: 4
    budget: 64
    index_of_master_SP: 1
    display: True
    scaling: null
    mode: serial
    M_update_scheme: median
    store_q_io: True

# coord:
#   c1:
#     index: 1
#     type: ADMM
#     beta: 1.1
#     gamma: 0.8
#     nsp: 4
#     budget: 500
#     index_of_master_SP: 1
#     display: True
#     scaling: null
#     mode: serial
#   M_update_scheme: max
#   store_q_io: True

subproblem:
  sp1:
    nv: 5
    index: 1
    vars: [v0, v1, v3, v4, v5]
    resps: [v2, v55]
    is_main: True
    MDA: 1
    coordinator: 1
    opt: SBJ_aircraft_opt
    type: callable
    fmin_nop: inf
    budget: 50
    display: False
    psize: 1.
    pupdate: DEFAULT
    freal: 34300
    solver: search
    save_results: True
    configurations: CSP

  sp2:
    nv: 2
    index: 2
    vars: [v8, v10]
    resps:  [v6,v7,v9, v56, v57]
    is_main: False
    MDA: 2
    coordinator: 1
    opt: SBJ_propulsion_opt
    type: callable
    fmin_nop: inf
    budget: 50
    display: False
    psize: 1.
    pupdate: DEFAULT
    solver: poll
    save_results: True
    configurations: CSP

  sp3:
    nv: 12
    index: 3
    vars: [v13,   v12,    v15, v17,  v18,      v19,   v20, v21,    v22,       v23, v24, v25 ]
    resps: [v16,v11,v14, v58, v59,v60]
    is_main: False
    MDA: 3
    coordinator: 1
    opt: SBJ_aerodynamics_opt
    type: callable
    fmin_nop: inf
    budget: 50
    display: False
    psize: 1.
    pupdate: DEFAULT
    solver: search
    save_results: True
    save_all_best: False
    configurations: CSP


  sp4:
    nv: 26
    index: 4
    vars: [v27,v30,v31, v32, v33, v34, v35, v36, v37,v38,v39,v40,v41,v42,v43,v44,v45, v46, v47,v48, v49, v50,v51,v52,v53,v54]
    resps: [v28,v29, v26]
    is_main: False
    MDA: 4
    coordinator: 1
    opt: SBJ_structure_opt
    type: callable
    fmin_nop: inf
    budget: 50
    display: False
    psize: 1.
    pupdate: DEFAULT
    solver: poll
    save_results: True
    save_all_best: False
    configurations: CSP

USER:
  h: 55000
  M: 1.4

MDO:
  architecture: IDF
  coordinator: 1
  subproblems: [1, 2, 3, 4]
  responses: [v2, v6,v7,v9, v16,v11,v14, v28,v29, v26]
  fmin: inf
  hmin: inf
  display: True
  inc_stop: 0.0000000001
  stop: "Iteration budget exhausted"
  tab_inc: []
  noprogress_stop: 50

# OPTIONS:
#   WORK_DIR: C:\Projects\SW\Dev\AB\DMDO_code\DEV\DMDO\tests\SBJ
CONFIG:
  CSP:
    search:
        "type": "sampling"
        "s_method": "ACTIVE"
        "ns": 10
        "visualize": False
"""

# Parse the YAML content
# You can load that content from an existing yaml file
basic_problem_definition = yaml.safe_load(yaml_content)

print("YAML Content Loaded:")
print(basic_problem_definition)
exec_modes = ["serial", "parallel"]
for em in exec_modes:
  bm_inputs_path = f"/content/sample_data/SBJ_{em}"

  if os.path.exists(bm_inputs_path):
    shutil.rmtree(bm_inputs_path)

  os.mkdir(bm_inputs_path)

  if os.path.exists(bm_inputs_path):
      shutil.rmtree(bm_inputs_path)

  os.mkdir(bm_inputs_path)
  with open(f'{bm_inputs_path}/SBJ.yaml', 'w') as file:
      file.write(yaml_content)
  with open(f'{bm_inputs_path}/SBJ.yaml', 'w') as file:
      file.write(yaml_content)

YAML Content Loaded:
{'variables': {'v0': ['SFC', 1, 2, 'fb', 1.0, 2.0, 4.0, 1, 'REAL'], 'v1': ['We', 1, 2, 'fb', 100.0, 15000.0, 30000.0, 1, 'REAL'], 'v2': ['Wt', 1, 3, 'ff', 5000.0, 25000.0, 100000.0, 1, 'REAL'], 'v3': ['LD', 1, 3, 'fb', 0.1, 5.0, 10.0, 1, 'REAL'], 'v4': ['Ws', 1, 4, 'fb', 5000.0, 25000.0, 100000.0, 1, 'REAL'], 'v5': ['Wf', 1, 4, 'fb', 5000.0, 25000.0, 100000.0, 1, 'REAL'], 'v6': ['SFC', 2, 1, 'ff', 1.0, 2.0, 4.0, 1, 'REAL'], 'v7': ['We', 2, 1, 'ff', 100.0, 15000.0, 30000.0, 1, 'REAL'], 'v8': ['D', 2, 3, 'fb', 1000.0, 40000.0, 70000.0, 1, 'REAL'], 'v9': ['ESF', 2, 3, 'ff', 0.5, 1.0, 1.5, 1, 'REAL'], 'v10': ['T', 2, None, 'u', 0.1, 0.6, 1.0, 1, 'REAL'], 'v11': ['D', 3, 2, 'ff', 1000.0, 40000.0, 70000.0, 1, 'REAL'], 'v12': ['ESF', 3, 2, 'fb', 0.5, 1.0, 1.5, 1, 'REAL'], 'v13': ['Wt', 3, 1, 'fb', 5000.0, 25000.0, 100000.0, 1, 'REAL'], 'v14': ['LD', 3, 1, 'ff', 0.1, 5.0, 10.0, 1, 'REAL'], 'v15': ['theta', 3, 4, 'fb', 0.2, 10.0, 50.0, 1, 'REAL'], 'v16': ['L', 3, 4, 'ff', 5

<>:297: SyntaxWarning: invalid escape sequence '\P'
<>:297: SyntaxWarning: invalid escape sequence '\P'
/tmp/ipykernel_94652/1019493531.py:297: SyntaxWarning: invalid escape sequence '\P'
  #   WORK_DIR: C:\Projects\SW\Dev\AB\DMDO_code\DEV\DMDO\tests\SBJ


<a id="disciplinary-analyses"></a>
#### SSBJ Disciplinary Analyses and Subproblems

The analytical formulation of the four disciplinary analysis are implemented below to calculate linked coupling variables, listed and defined in Table 1.
<center>

|Discipline                 | Subproblem Index  | Inputs                                                                                                                                                          | Output                                                            | Python Class       |
|---------------------------|-------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------|-------------------------------------------------------------------|---------------------------------------------------------------|
|Aircraft                   | 1           | $\text{SFC}$, $W_e$, $L/D$ $W_s$, $W_f$, $h$, $M$,                                                                                                              | $W_t$, $\text{range}$     		                                | `SSBJ_Aircraft`                                                      |
|Propulsion 	            | 2           | $T$, $D$, $h$, $M$,                                                                                                                                             | $\text{SFC}$, $W_e$, $\text{ESF}$, $T_{E}$, $\text{Throttle}$     | `SSBJ_propulsion`                                                      |
|Aerodynamics 	            | 3           | $W_t$, $\text{ESF}$, $\theta$, $t/c$, $AR_w$, $\Lambda_w$, $S_\text{ref}$, $S_\text{ht}$, $AR_\text{ht}$, $\Lambda_\text{ht}$, $L_w$, $L_\text{ht}$, $h$, $M$   | $L$, $D$, $L/D$, $P_g$, $CLo_1$, $CLo_2$                          | `SSBJ_Aerodynamics`                                                      |
|Structures 		        | 4           | $L$, $t/c$, $AR_w$, $\Lambda_w$, $S_\text{ref}$, $S_\text{ht}$, $AR_\text{ht}$, $\lambda$, $\mathbf{t}$, $\mathbf{t}_s$, $h$, $M$,                              | $W_s$, $W_f$, $\theta$, $\mathbf{g}_1$                            | `SSBJ_Structures`                                                      |

</center>
<center>Table.1 Definition of disciplinary analysis files</center>


##### *Aircraft Performance*
# <a id="aircraft-performance"></a>

The aircraft subproblem is essentially the system-level integrator. It ties together outputs from propulsion, structures, and aerodynamics disciplines into overall aircraft performance and feasibility. Think of it as the place where all disciplinary simulation outputs are employed to evaluate mission and design requirements/criteria.

The aircraft subproblem does not compute everything per se. It depends on disciplinary response outputs and candidate solutions computed by other disciplinary subproblems and analyses.

<center>


| Discipline   | Input to Aircraft |
| ------------ | ----------------- |
| Aerodynamics | ( $C_L$, $C_D$ )      |
| Structures   | Weight ( $W_s$ )    |
| Propulsion   | Thrust ($T$), $SFC$       |

</center>


Linked variables are inherently inconsistent at the start of coordination process, so the Aircraft subproblem enforces local variables copy to be consistent with their linked pairs during the subproblem optimization process.

The aircraft performance subproblem formulation is given below. Superscripts denote the subproblem at which variable values are computed: ``a`` refers to aircraft, ``p`` to propulsion, ``ae`` to aerodynamics, and ``s`` to structures.

\begin{equation*}
	\begin{aligned}
		& \underset{\mathbf{x}}{\text{minimize}}
		& & W_{\mathrm{t}}^{\mathrm{a}}+\phi\left(\mathrm{SFC}^{\mathrm{a}}-\mathrm{SFC}^{\mathrm{p}}\right)+\phi\left(W_{\mathrm{e}}^{\mathrm{a}}-W_{\mathrm{e}}^{\mathrm{p}}\right)\\
    & & & +\phi\left(L / D^{\mathrm{a}}-L / D^{\mathrm{ae}}\right)+\phi\left(W_{\mathrm{t}}^{\mathrm{ae}}-W_{\mathrm{t}}^{\mathrm{a}}\right)+\phi\left(W_{\mathrm{s}}^{\mathrm{a}}-W_{\mathrm{s}}^{\mathrm{s}}\right)+\phi\left(W_{\mathrm{f}}^{\mathrm{a}}-W_{\mathrm{f}}^{\mathrm{s}}\right) \\
		& \text{subject to}
		& & g_{\text {aircraft }}\left(\mathrm{SFC}^{\mathrm{a}}, W_{\mathrm{e}}^{\mathrm{a}}, L / D^{\mathrm{a}}, W_{\mathrm{f}}^{\mathrm{a}}, W_{\mathrm{s}}^{\mathrm{a}}\right) \leq \mathbf{0} \\
    & \text{while solving}
		& & W_{\mathrm{t}}^{\mathrm{a}}=W_{\mathrm{t}}\left(W_{\mathrm{e}}^{\mathrm{a}}, W_{\mathrm{f}}^{\mathrm{a}}, W_{\mathrm{s}}^{\mathrm{a}}\right)\\
    & \text{where}
		& & \mathbf{x} = \left[\mathrm{SFC}^{\mathrm{a}}, W_{\mathrm{e}}^{\mathrm{a}}, L / D^{\mathrm{a}}, W_{\mathrm{s}}^{\mathrm{a}}, W_{\mathrm{f}}^{\mathrm{a}}\right]^\textit{T}
	\end{aligned}
\end{equation*}


The objective/coupling variable $W_{\mathrm{t}}^{\mathrm{a}}$ is calculated by  `SBJ_Aircraft`. The constraint $g_{\mathrm{aircraft}}$ and the overall optimization problem are defined in `SBJ_aircraft_opt`

In [ ]:
#@title #####*Aircraft Performance Class: Analysis and Subproblem Functions*
from typing import List

import numpy as np
import math

class SSBJ_Aircraft:
    """Class representing a Supersonic Business Jet (SSBJ) aircraft with performance calculations."""

    def __init__(self, h=55000, Mach=1.4, SFCp=2, We=15000, LDr=5.0, Ws=25000, Wf=25000):
        """Initialize the SSBJ aircraft with given parameters.

        Args:
            h (float): Altitude in feet
            Mach (float): Mach number
            SFCp (float): Specific fuel consumption (per hour)
            We (float): Engine weight in pounds
            LDr (float): Lift-to-drag ratio
            Ws (float): Structural weight in pounds
            Wf (float): Fuel weight in pounds
        """
        self.h = h
        self.Mach = Mach
        self.SFCp = SFCp
        self.We = We
        self.LDr = LDr
        self.Ws = Ws
        self.Wf = Wf

        # Calculate total weight


    def _calculate_theta_r(self) -> float:
        """Calculate the temperature ratio based on altitude.

        Returns:
            float: Temperature ratio theta_r
        """
        if self.h < 36089:
            return 1 - 0.000006875 * self.h
        else:
            return 0.7519

    def _calculate_range(self, Wt, theta_r) -> float:
        """Calculate the aircraft range using the Breguet range equation.

        Returns:
            float: Range in nautical miles
        """
        return (self.Mach * self.LDr * 661.0 *
                np.sqrt(theta_r / self.SFCp) *
                math.log(Wt / (Wt - self.Wf)))

    def _calculate_constraints(self, range) -> List[float]:
        """Calculate the constraint value for the optimization problem.

        Returns:
            float: Constraint value g
        """
        return [-range / 2000.0 + 1.0]

    def get_results(self) -> dict:
        """Get all calculated results as a dictionary.

        Returns:
            dict: Dictionary containing all calculated values
        """
        return {
            "altitude": self.h,
            "Mach": self.Mach,
            "SFCp": self.SFCp,
            "engine_weight": self.We,
            "lift_drag_ratio": self.LDr,
            "structural_weight": self.Ws,
            "fuel_weight": self.Wf,
            "total_weight": self.SBJ_aircraft_analysis()[0],
            "temperature_ratio": self._calculate_theta_r(),
            "range": self.SBJ_aircraft_analysis()[1],
            "constraint": self._calculate_constraints(range=self.SBJ_aircraft_analysis()[1])
        }

    def __str__(self) -> str:
        """String representation of the aircraft.

        Returns:
            str: Formatted string with key parameters
        """
        return (f"SSBJ Aircraft (h={self.h} ft, Mach={self.Mach}, Wt={self.Wt} lb, "
                f"range={self.range:.1f} nm, violation={sum(self.g):.3f})")

    def __repr__(self) -> str:
        """String representation for debugging.

        Returns:
            str: Detailed string representation
        """
        return (f"SSBJ_Aircraft(h={self.h}, Mach={self.Mach}, SFCp={self.SFCp}, "
                f"We={self.We}, LDr={self.LDr}, Ws={self.Ws}, Wf={self.Wf})")

    def SBJ_aircraft_analysis(self):
        Wt = self.We + self.Wf + self.Ws

        # Calculate theta_r based on altitude
        theta_r = self._calculate_theta_r()

        # Calculate range
        range = self._calculate_range(Wt=Wt, theta_r = theta_r)

        return [Wt, range]

    def SBJ_aircraft_opt(self, Wt, range):
        return [Wt, self._calculate_constraints(range=range)]

# Example usage and testing
if __name__ == "__main__":
    # Create an instance with default values
    aircraft = SSBJ_Aircraft()

    # Print results
    # print(aircraft)
    # print(f"Range: {aircraft.range:.1f} nautical miles")
    # print(f"Constraint violation: {sum(aircraft.g):.3f}")

    # Get all results as dictionary
    results = aircraft.get_results()
    for key, value in results.items():
        print(f"{key}: {value}")

altitude: 55000
Mach: 1.4
SFCp: 2
engine_weight: 15000
lift_drag_ratio: 5.0
structural_weight: 25000
fuel_weight: 25000
total_weight: 65000
temperature_ratio: 0.7519
range: 1377.4021917957266
constraint: [np.float64(0.3112989041021367)]


In [ ]:
#@title #####*Aircraft Performance Optimization Subproblem test*
import platform
from typing import Dict
from OMADS import poll, mads, search
import pandas # Added to address the numexpr issue

pandas.options.compute.use_numexpr = False # Added to address the numexpr issue

def SBJ_aircraft_opt_test(x):
  ssbj_aircraft = SSBJ_Aircraft(We=x[1], Ws=x[3], Wf=x[4], SFCp=x[0], LDr=x[2])
  Wt, range = ssbj_aircraft.SBJ_aircraft_analysis()
  return ssbj_aircraft.SBJ_aircraft_opt(Wt=Wt, range=range)

def test_SBJ_aircraft_performance():

  # Rest of your code follows
  is_mac = platform.platform().split('-')[0] == 'macOS'
  lb = [1.,100.,5000.,0.1,5000.,5000.]
  ub = [4.,30000,100000,10.,100000.,100000.]
  bl = [2.,15000.,25000.,5.,25000.,25000.]
  vnames = ["SFC", "We", "Wt", "LD", "Ws", "Wf"]
  vartype = ["REAL"] * 6
  constants = None
  constants_name = None
  wd = os.path.dirname("/content/sample_data/")

  param = {"baseline": bl,
           "lb": lb,
           "ub": ub,
           "var_names": vnames,
           "var_type": vartype,
           "var_sets": None,
           "scaling": [1] * len(lb),
           # "post_dir": "./post",
           "constants": constants,
           "constants_name": constants_name,
           "name": "SBJ_Aircraft",
           "mesh_type": "GMESH",
           "post_dir": os.path.join(wd, "test_SBJ_post"),
           "constraints_type": ["PB"],
           "lambda_multipliers": [1000],
           "rho": 0.01,
           "h_max": np.inf,

           }

  options = {
      "seed": 10000,
      "budget": 150,
      "tol": 1E-12,
      "psize_init": 1,
      "display": False,
      "opportunistic": False,
      "check_cache": True,
      "store_cache": True,
      "collect_y": False,
      "rich_direction": True,
      "precision": "high" if is_mac else "medium",
      "save_results": False,
      "save_coordinates": False,
      "save_all_best": False,
      "parallel_mode": False
  }
  # isWin = platform.platform().split('-')[0] == 'Windows'
  # options["precision"] = "high" if isWin else "medium"

  search_dict = {
              "type": "sampling",
              "s_method": "ACTIVE",
              "ns": 50,
              "visualize": False
  }

  data = {"evaluator": {"blackbox": SBJ_aircraft_opt_test}, "param": param,
          "options": options, "search": search_dict}

  h = 55000
  M = 1.4

  out_mads, _, _ = mads.main(data)
  print("=== Testing Structural Wing Subproblem: Minimum Candidate")
  print(f"xmin={out_mads["xmin"]}")
  print(f"fmin_feasible={out_mads["fmin"]}")
  print(f"hmin_feasible={out_mads["hmin"]}")

test_SBJ_aircraft_performance()


SBJ_Aircraft: |Pending:  █  , Feasible:  █  , Infeasible:  █  |█████████████████████████████████████████████████| 23/150 #Evaluated
SBJ_Aircraft: |Pending:  █  , Feasible:  █  , Infeasible:  █  |█████████████████████████████████████████████████| 41/150 #Evaluated
SBJ_Aircraft: |Pending:  █  , Feasible:  █  , Infeasible:  █  |█████████████████████████████████████████████████| 58/150 #Evaluated
SBJ_Aircraft: |Pending:  █  , Feasible:  █  , Infeasible:  █  |█████████████████████████████████████████████████| 74/150 #Evaluated
SBJ_Aircraft: |Pending:  █  , Feasible:  █  , Infeasible:  █  |█████████████████████████████████████████████████| 91/150 #Evaluated
frameCenter = 5.0, δ = 0.01 : it gave                   f = [], h = inf, x= [2.0, 100.0, 5000.0, 0.09999999999999964, 25000.0, 25000.0] which is still lower than 0.1
frameCenter = 5.0, δ = 0.01 : it gave                   f = [], h = inf, x= [2.0, 15000.0, 5000.0, 0.09999999999999964, 5000.0, 25000.0] which is still lower than 0.1
frameCe

##### *Propulsion Subproblem*

The Propulsion subproblem in the SBJ distributed MDO benchmark focuses on modeling the engine's ability to produce thrust efficiently while remaining consistent with the aircraft's mission requirements. It is a tightly coupled discipline because its outputs directly affect range, drag balance, and fuel usage.

The propulsion subproblem is usually not optimizing a standalone engineering goal like in real engine design. Instead, it minimizes inconsistency with target variables or contributes indirectly to:

* Minimizing fuel consumption
* Meeting aircraft-level objectives

The propulsion disciplinary subproblem formulation is given below:

\begin{equation*}
	\begin{aligned}
		& \underset{\mathbf{x}}{\text{minimize}}
		& & \phi\left(\mathrm{SFC}^{\mathrm{a}}-\mathrm{SFC}^{\mathrm{p}}\right)+\phi\left(W_{\mathrm{e}}^{\mathrm{a}}-W_{\mathrm{e}}^{\mathrm{p}}\right)+\phi\left(D^{\mathrm{p}}-D^{\mathrm{ae}}\right)+\phi\left(\mathrm{ESF}^{\mathrm{ae}}-\mathrm{ESF}^{\mathrm{p}}\right) \\
    & \text{subject to}
		& & \mathbf{g}_{\text {prop }}\left(D^{\mathrm{p}}, T\right) \leq \mathbf{0} \\
    & \text{while solving}
		& & W_{\mathrm{e}}^{\mathrm{p}}=W_{\mathrm{e}}\left(D^{\mathrm{p}}, T\right) \\
    & & & \mathrm{SFC}^{\mathrm{p}}=\operatorname{SFC}\left(D^{\mathrm{p}}, T\right) \\
    & & & \mathrm{ESF}^{\mathrm{p}}=\operatorname{ESF}\left(D^{\mathrm{p}}, T\right) \\
    & \text{where}
		& & \mathbf{x} = \left[D^{\mathrm{p}}, T\right]^\textit{T}
	\end{aligned}
\end{equation*}

The calculation of the coupling variables $W_{\mathrm{e}}^{\mathrm{p}}$, $\mathrm{SFC}^{\mathrm{p}}$, and $\mathrm{ESF}^{\mathrm{p}}$, is given by `SBJ_propulsion_analysis`. The calculation of the objective and constraint and definition of the optimization problem is given by `SBJ_propulsion_opt`.

In [ ]:
#@title #####*Propulsion Class: Analysis and Subproblem Functions*
import copy

import numpy as np


class SSBJPropulsion:
    """Class for SSBJ propulsion system calculations."""

    def __init__(self, D=40000, T=0.6, h=55000, Mach=1.4):
        """Initialize constants and default values."""
        # Constants from original paper
        self.C = [500.0, 16000.0, 4.0, 4360.0, 0.01375, 1.0]
        self.Wbe = self.C[3]  # constant weight [lbs]

        # Polynomial coefficient matrix
        self.R = np.array([
            [0.2736, 0.3970, 0.8152, 0.9230, 0.1108],
            [0.4252, 0.4415, 0.6357, 0.7435, 0.1138],
            [0.0329, 0.8856, 0.8390, 0.3657, 0.0019],
            [0.0878, 0.7248, 0.1978, 0.0200, 0.0169],
            [0.8955, 0.4568, 0.8075, 0.9239, 0.2525]
        ])

        # Throttle scaling factor
        self.throttle_scale = 16168

        # Initial values
        self.h = 55000
        self.Mach = 1.4
        self.Drag = D
        self.Throttle = T

        # SFC coefficients
        self.s = [1.13238425638512, 1.53436586044561, -0.00003295564466,
                 -0.00016378694115, -0.31623315541888, 0.00000410691343,
                 -0.00005248000590, -0.00000000008574, 0.00000000190214,
                 0.00000001059951]

        # Throttle coefficients
        self.p = [11483.7822254806, 10856.2163466548, -0.5080237941,
                 3200.157926969, -0.1466251679, 0.0000068572]

        # Constraint values
        self.Temp_uA = 1.02
        self.Throttle_uA = 16168 * 0.6

    def poly_approx(self, S, S_new, flag, S_bound):
        """Calculate polynomial approximation for propulsion parameters."""
        S_norm = []
        S_shifted = []
        Ai = []
        Aij = np.zeros((len(S), len(S)))

        for i in range(len(S)):
            S_norm.append(S_new[i] / S[i])
            if S_norm[i] > 1.25:
                S_norm[i] = 1.25
            elif S_norm[i] < 0.75:
                S_norm[i] = 0.75

            S_shifted.append(S_norm[i] - 1)
            a = 0.1
            b = a

            if flag[i] == 5:
                # CALCULATE POLYNOMIAL COEFFICIENTS (S-ABOUT ORIGIN)
                So = 0
                Sl = So - S_bound[i]
                Su = So + S_bound[i]
                Mtx_shifted = np.array([[1, Sl, Sl**2], [1, So, So**2], [1, Su, Su**2]])

                F_bound = np.array([1 + (.5*a)**2, 1, 1 + (.5*b)**2])
                A = np.linalg.solve(Mtx_shifted, F_bound)
                Ao = A[0]
                Ai.append(A[1])
                Aij[i,i] = A[2]
            else:
                if flag[i] == 0:
                    S_shifted.append(0)
                elif flag[i] == 3:
                    a *= -1
                    b = copy.deepcopy(a)
                elif flag[i] == 2:
                    b = 2 * a
                elif flag[i] == 4:
                    a *= -1
                    b = 2*a

                # DETERMINE BOUNDS ON FF DEPENDING ON SLOPE-SHAPE
                # CALCULATE POLYNOMIAL COEFFICIENTS (S-ABOUT ORIGIN)
                So = 0
                Sl = So - S_bound[i]
                Su = So + S_bound[i]
                Mtx_shifted = np.array([[1, Sl, Sl**2], [1, So, So**2], [1, Su, Su**2]])
                F_bound = np.array([1 - .5*a, 1, 1 + .5*b])
                A = np.linalg.solve(Mtx_shifted, F_bound)
                Ao = A[0]
                Ai.append(A[1])
                Aij[i,i] = A[2]

        # Calculate cross terms
        for i in range(len(S)):
            for j in range(i+1, len(S)):
                Aij[i, j] = Aij[i,i] * self.R[i,j]
                Aij[j, i] = Aij[i, j]

        S_shifted = np.array(S_shifted)

        # Calculate FF (Performance Factor)
        FF = Ao + np.dot(Ai, (np.transpose(S_shifted))) + \
             (1/2) * np.dot(np.dot(S_shifted, Aij), np.transpose(S_shifted))

        return FF

    def calculate_sfc(self, Mach, h, Throttle):
        """Calculate Specific Fuel Consumption (SFC)."""
        return (self.s[0] + self.s[1]*Mach + self.s[2]*h + self.s[3]*Throttle + \
                self.s[4]*Mach**2 + 2*h*Mach*self.s[5] + 2*Throttle*Mach*self.s[6] + \
                self.s[7]*h**2 + 2*Throttle*h*self.s[8] + self.s[9]*Throttle**2)

    def calculate_esf(self, Thrust, Throttle):
        """Calculate Engine Specific Force (ESF)."""
        return (Thrust / 2) / Throttle

    def calculate_engine_weight(self, ESFp):
        """Calculate engine weight based on ESF."""
        return self.Wbe * (ESFp**1.05) * 2

    def calculate_tempe_throttleua(self):
        """ Calculate Temp_E and  Throttle_uA """
        Temp_E = self.poly_approx([self.Mach, self.h, self.Drag], [self.Mach, self.h, self.Throttle], [2, 4, 2],\
                                   [.25, .25, .25])

        Throttle_uA = self.p[0] + self.p[1]*self.Mach + self.p[2]*self.h + self.p[3]*self.Mach**2 + \
                     2*self.p[4]*self.Mach*self.h + self.p[5]*self.h**2
        return Temp_E, Throttle_uA

    def calculate_constraints(self, Temp_E, Throttle_uA):
        """Calculate constraint functions for the propulsion system."""
        Dim_Throttle = self.Throttle * self.throttle_scale
        # SFCp = self.calculate_sfc(self.Mach, self.h, Dim_Throttle)

        # # Calculate ESF
        # ESFp = self.calculate_esf(self.Drag, Dim_Throttle)
        # Temp_E = self.poly_approx([Mach, h, Throttle], [Mach, h, Throttle], [2, 4, 2], [.25, .25, .25])

        # Throttle_uA = self.p[0] + self.p[1]*Mach + self.p[2]*h + self.p[3]*Mach**2 + \
        #              2*self.p[4]*Mach*h + self.p[5]*h**2

        g1 = Temp_E / self.Temp_uA - 1
        g2 = Dim_Throttle / Throttle_uA - 1
        return [g1, g2]


    def SBJ_propulsion_analysis(self):
        """Run complete analysis with current parameters."""
        Dim_Throttle = self.Throttle * self.throttle_scale
        # Calculate SFC
        SFCp = self.calculate_sfc(self.Mach, self.h, Dim_Throttle)

        # Calculate ESF
        ESFp = self.calculate_esf(self.Drag, Dim_Throttle)

        # Calculate engine weight
        We = self.calculate_engine_weight(ESFp)

        # Print results
        # print(f'SFCp = {SFCp}')
        # print(f'ESFp = {ESFp}')
        # print(f'We = {We}')
        # print(f'g1 = {g1}')
        # print(f'g2 = {g2}')
        Temp_E, Throttle_uA = self.calculate_tempe_throttleua()
        return [SFCp, We, ESFp, Temp_E, Throttle_uA]

    def SBJ_propulsion_opt(self, Temp_E, Throttle_uA):
        return [0, self.calculate_constraints(Temp_E, Throttle_uA)]

    def print_results(self):
        SFCp, We, ESFp, Temp_E, Throttle_uA = self.SBJ_propulsion_analysis()
        G = self.calculate_constraints(Temp_E, Throttle_uA)
        print('SFCp = ', SFCp)
        print('ESFp = ', ESFp)
        print('Temp_E = ', Temp_E)
        print('Throttle_uA = ', Throttle_uA)
        print('Dim_Throttle = ', self.Throttle * self.throttle_scale)
        print('We = ', We)
        print('g1 = ', G[0])
        print('g2 = ', G[1])



if __name__ == "__main__":
    prop = SSBJPropulsion()
    prop.print_results()

SFCp =  1.23410448954172
ESFp =  2.061685634174501
Temp_E =  0.9375
Throttle_uA =  3176.2401155565603
Dim_Throttle =  9700.8
We =  18640.17785893877
g1 =  -0.08088235294117652
g2 =  2.0541771550858225


##### *Structural Analysis of the Aircraft Wings*

The Structures subproblem in the SBJ distributed MDO benchmark is responsible for estimating the airframe structural weight and ensuring the design can withstand aerodynamic loads. It plays a central role because weight directly drives almost every aircraft-level metric (range, fuel, lift requirements).

The structures module acts as:

* A weight estimator → provides structural weight to the Aircraft subproblem
* A feasibility checker → ensures stresses or loads remain within limits
* A coupling participant → tightly linked with Aerodynamics and Aircraft

It receives loads (e.g., lift) and geometry, then returns a consistent structural response.

The structural subproblem formulation is given below:

\begin{equation*}
	\begin{aligned}
		& \underset{\mathbf{x}}{\text{minimize}}
		& & \phi\left(\mathrm{ESF}^{\mathrm{ae}}-\mathrm{ESF}^{\mathrm{p}}\right)+\phi\left(D^{\mathrm{p}}-D^{\mathrm{ae}}\right)+\phi\left(W_{\mathrm{t}}^{\mathrm{ae}}-W_{\mathrm{t}}^{\mathrm{a}}\right)+\phi\left(L / D^{\mathrm{a}}-L / D^{\mathrm{ae}}\right) \\
    & & & +\phi\left(L^{\mathrm{s}}-L^{\mathrm{ae}}\right)+\phi\left(\theta^{\mathrm{ae}}-\theta^{\mathrm{s}}\right)+\phi\left(t / c^{\mathrm{ae}}-t / c^{\mathrm{s}}\right)+\phi\left(\mathrm{AR}_{\mathrm{w}}^{\mathrm{ae}}-\mathrm{AR}_{\mathrm{w}}^{\mathrm{s}}\right) \\
    & & & +\phi\left(\Lambda_{\mathrm{w}}^{\mathrm{ae}}-\Lambda_{\mathrm{w}}^{\mathrm{s}}\right)+\phi\left(S_{\mathrm{ref}}^{\mathrm{ae}}-S_{\mathrm{ref}}^{\mathrm{s}}\right)+\phi\left(S_{\mathrm{ht}}^{\mathrm{ae}}-S_{\mathrm{ht}}^{\mathrm{s}}\right)+\phi\left(\mathrm{AR}_{\mathrm{ht}}^{\mathrm{ae}}-\mathrm{AR}_{\mathrm{ht}}^{\mathrm{s}}\right) \\
    & \text{subject to}
		& & \mathbf{g}_{\text {aero }}\left(W_{\mathrm{t}}^{\mathrm{ae}}, \theta^{\mathrm{ae}}, \mathrm{ESF}^{\mathrm{ae}}, t / c^{\mathrm{ae}}, \mathrm{AR}_{\mathrm{w}}^{\mathrm{ae}}, \Lambda_{\mathrm{w}}^{\mathrm{ae}}, S_{\mathrm{ref}}^{\mathrm{ae}}, S_{\mathrm{ht}}^{\mathrm{ae}}, \mathrm{AR}_{\mathrm{ht}}^{\mathrm{ae}}, \Lambda_{\mathrm{ht}}, L_{\mathrm{w}}, L_{\mathrm{ht}}\right) \leq \mathbf{0} \\
    & \text{while solving}
		& & D^{\mathrm{ae}}=D\left(W_{\mathrm{t}}^{\mathrm{ae}}, \theta^{\mathrm{ae}}, \mathrm{ESF}^{\mathrm{ae}}, t / c^{\mathrm{ae}}, \mathrm{AR}_{\mathrm{w}}^{\mathrm{ae}}, \Lambda_{\mathrm{w}}^{\mathrm{ae}}, S_{\mathrm{ref}}^{\mathrm{ae}}, S_{\mathrm{ht}}^{\mathrm{ae}}, \mathrm{AR}_{\mathrm{ht}}^{\mathrm{ae}}, \Lambda_{\mathrm{ht}}, L_{\mathrm{w}}, L_{\mathrm{ht}}\right) \\
    & & & L / D^{\mathrm{ae}}=L / D\left(W_{\mathrm{t}}^{\mathrm{ae}}, \theta^{\mathrm{ae}}, \mathrm{ESF}^{\mathrm{ae}}, t / c^{\mathrm{ae}}, \mathrm{AR}_{\mathrm{w}}^{\mathrm{ae}}, \Lambda_{\mathrm{w}}^{\mathrm{ae}}, S_{\mathrm{ref}}^{\mathrm{ae}}, S_{\mathrm{ht}}^{\mathrm{ae}}, \mathrm{AR}_{\mathrm{ht}}^{\mathrm{ae}}, \Lambda_{\mathrm{ht}}, L_{\mathrm{w}}, L_{\mathrm{ht}}\right) \\
    & & & L^{\mathrm{ae}}=L\left(W_{\mathrm{t}}^{\mathrm{ae}}, \theta^{\mathrm{ae}}, \mathrm{ESF}^{\mathrm{ae}}, t / c^{\mathrm{ae}}, \mathrm{AR}_{\mathrm{w}}^{\mathrm{ae}}, \Lambda_{\mathrm{w}}^{\mathrm{ae}}, S_{\mathrm{ref}}^{\mathrm{ae}}, S_{\mathrm{ht}}^{\mathrm{ae}}, \mathrm{AR} \mathrm{ht}_{\mathrm{ht}}^{\mathrm{ae}}, \Lambda_{\mathrm{ht}}, L_{\mathrm{w}}, L_{\mathrm{ht}}\right) \\
    & \text{where}
		& & \mathbf{x} = \left[\mathrm{ESF}^{\mathrm{ae}}, W_{\mathrm{t}}^{\mathrm{ae}}, \theta^{\mathrm{ae}}, t / c^{\mathrm{ae}}, \mathrm{AR}_{\mathrm{w}}^{\mathrm{ae}}, \Lambda_{\mathrm{w}}^{\mathrm{ae}}, S_{\mathrm{ref}}^{\mathrm{ae}}, S_{\mathrm{ht}}^{\mathrm{ae}}, \mathrm{AR}_{\mathrm{ht}}^{\mathrm{ae}}, \Lambda_{\mathrm{ht}}, L_{\mathrm{w}}, L_{\mathrm{ht}}\right]^\textit{T}
	\end{aligned}
\end{equation*}

There is no local objective. The calculation of the coupling variables $D^{\mathrm{ae}}$, $L / D^{\mathrm{ae}}$, and $L^{\mathrm{ae}}$, is given by `calculate_structural_response`. The calculation of the constraints $\mathbf{g}_{\text {prop }}$ and definition of the optimization problem is given by `SBJ_structure_opt`.

In [ ]:
#@title #####*Structural Wing CLass: Analysis and Subproblem Functions*

class WingDesignAnalyzer:
    """
    A class to analyze and design aircraft wings based on structural and aerodynamic parameters.
    """

    def __init__(self, h=55000, Mach=1.4, tc=0.0641, ARw=2.5, LAMBDAw=70.0, Sref=667, Sht=99.7,
                 ARht=5.5, lambdatr=0.1, Lift=34300, t=[0.97, 0.52, 0.21, 4.00, 3.66, 0.91, 0.97, 0.52, 0.21],
                 ts=[2.17, 1.48, 0.76, 4.40, 4.02,1.01, 2.17, 1.48, 0.76]):
        # Inputs
        self.h = h
        self.Mach = Mach
        self.tc = tc
        self.ARw = ARw
        self.LAMBDAw = LAMBDAw
        self.Sref = Sref
        self.Sht = Sht
        self.ARht = ARht
        self.lambdatr = lambdatr
        self.Lift = Lift

        # Constants
        self.C = [500.0, 16000.0, 4.0, 4360.0, 0.01375, 1.0]
        self.G = 4000000 * 144
        self.E = 10600000 * 144
        self.nu = 0.3
        self.rho_alum = 0.1 * 144
        self.rho_core = 0.1 * 144 / 10
        self.rho_fuel = 6.5 * 7.4805
        self.Fw_at_t = 5
        self.k = 6.09375

        # Local variables
        self.Z = [self.tc, self.h, self.Mach, self.ARw, self.LAMBDAw, self.Sref, self.Sht, self.ARht]
        self.LAMBDA = self.lambdatr
        self.L = self.Lift

        # Panel parameters
        self.NP = 9  # number of panels per halfspan
        self.n = 90
        self.rn = int(self.n / self.NP)

        # Thickness values (converted from inches to feet)
        self.ti = t
        self.tsi = ts
        self.t = np.array([ti / 12.0 for ti in self.ti])
        self.ts = np.array([tsi / 12.0 for tsi in self.tsi])

        # Split thickness arrays
        self.t1 = self.t[:3]
        self.t2 = self.t[3:6]
        self.t3 = self.t[6:9]
        self.ts1 = self.ts[:3]
        self.ts2 = self.ts[3:6]
        self.ts3 = self.ts[6:9]

        # Beta factor
        self.beta = 0.9

        # Initialize results
        self.c = None
        self.c_box = None
        self.Sweep_40 = None
        self.D_mx = None
        self.b = None
        self.a = None
        self.P = None
        self.Mz = None
        self.Mx = None
        self.bend_twist = None
        self.Spanel = None
        self.Phi = None
        self.twist = None
        self.deltaL_divby_q = None
        self.Wtop_alum = None
        self.Wbottom_alum = None
        self.Wside_alum = None
        self.Wtop_core = None
        self.Wbottom_core = None
        self.Wside_core = None
        self.W_wingstruct = None
        self.W_fuel_wing = None
        self.Bh = None
        self.W_ht = None
        self.Wf = None
        self.Ws = None
        self.theta = None
        # self.G = None
        self.G1 = None

    def polyApprox(self, S, S_new, flag, S_bound):
        """Calculate polynomial approximation for structural parameters."""
        S_norm = []
        S_shifted = []
        Ai = []
        Aij = np.zeros((len(S), len(S)))

        for i in range(len(S)):
            S_norm.append(S_new[i] / S[i])
            if S_norm[i] > 1.25:
                S_norm[i] = 1.25
            elif S_norm[i] < 0.75:
                S_norm[i] = 0.75
            S_shifted.append(S_norm[i] - 1)
            a = 0.1
            b = a

            if flag[i] == 5:
                # Calculate polynomial coefficients (S-about origin)
                So = 0
                Sl = So - S_bound[i]
                Su = So + S_bound[i]
                Mtx_shifted = np.array([[1, Sl, Sl**2], [1, So, So**2], [1, Su, Su**2]])

                F_bound = np.array([1 + (.5 * a)**2, 1, 1 + (.5 * b)**2])
                A = np.linalg.solve(Mtx_shifted, F_bound)
                Ao = A[0]
                Ai.append(A[1])
                Aij[i, i] = A[2]
            else:
                if flag[i] == 0:
                    S_shifted.append(0)
                elif flag[i] == 3:
                    a *= -1.
                    b = copy.deepcopy(a)
                elif flag[i] == 2:
                    b = 2 * a
                elif flag[i] == 4:
                    a *= -1
                    b = 2 * a

                # Determine bounds on FF depending on slope-shape
                So = 0
                Sl = So - S_bound[i]
                Su = So + S_bound[i]
                Mtx_shifted = np.array([[1, Sl, Sl**2], [1, So, So**2], [1, Su, Su**2]])
                F_bound = np.array([1 - .5 * a, 1, 1 + .5 * b])
                A = np.linalg.solve(Mtx_shifted, F_bound)
                Ao = A[0]
                Ai.append(A[1])
                Aij[i, i] = A[2]

        # Create correlation matrix R
        R = np.array([[0.2736, 0.3970, 0.8152, 0.9230, 0.1108],
                      [0.4252, 0.4415, 0.6357, 0.7435, 0.1138],
                      [0.0329, 0.8856, 0.8390, 0.3657, 0.0019],
                      [0.0878, 0.7248, 0.1978, 0.0200, 0.0169],
                      [0.8955, 0.4568, 0.8075, 0.9239, 0.2525]])

        # Fill in correlation matrix
        for i in range(len(S)):
            for j in range(i+1, len(S)):
                Aij[i, j] = Aij[i, i] * R[i, j]
                Aij[j, i] = Aij[i, j]

        S_shifted = np.array(S_shifted)

        # Calculate FF (structural efficiency factor)
        FF = Ao + np.dot(Ai, S_shifted.T) + 0.5 * np.dot(np.dot(S_shifted, Aij), S_shifted.T)

        return FF

    def Wing_Mod(self, Z, LAMBDA):
        """Calculate wing geometry parameters."""
        c = [0, 0, 0, 0]
        x = [0] * 8
        y = [0] * 8

        b = max(2, np.real(np.sqrt(Z[3] * Z[5])))
        c[0] = 2 * Z[5] / ((1 + LAMBDA) * b)
        c[3] = LAMBDA * c[0]
        x[0] = 0
        y[0] = 0
        x[1] = c[0]
        y[1] = 0
        x[6] = (b / 2) * np.tan(Z[4] * np.pi / 180)
        y[6] = b / 2
        x[7] = x[6] + c[3]
        y[7] = b / 2
        y[2] = b / 6
        x[2] = (x[6] / y[6]) * y[2]
        y[4] = b / 3
        x[4] = (x[6] / y[6]) * y[4]
        x[5] = x[7] + ((x[1] - x[7]) / y[7]) * (y[7] - y[4])
        y[5] = y[4]
        x[3] = x[7] + ((x[1] - x[7]) / y[7]) * (y[7] - y[2])
        y[3] = y[2]
        c[1] = x[3] - x[2]
        c[2] = x[5] - x[4]
        TE_sweep = (np.arctan((x[7] - x[1]) / y[7])) * 180 / np.pi
        Sweep_40 = (np.arctan(((x[7] - 0.6 * (x[7] - x[6])) - 0.4 * x[1]) / y[7])) * 180 / np.pi

        l = np.multiply([c[i] for i in range(3)], 0.4 * np.cos(Z[4] * np.pi / 180))  # noqa: E741
        k = np.multiply([c[i] for i in range(3)], 0.6 * np.sin((90 - TE_sweep) * np.pi / 180) /
                       np.sin((90 + TE_sweep - Z[4]) * np.pi / 180))
        c_box = np.add(l, k)
        D_mx = np.subtract(l, np.multiply(0.407, c_box))

        return c, c_box, Sweep_40, D_mx, b, l

    def loads(self, b, c, Sweep_40, D_mx, L, Izz, Z, E):
        """Calculate load distribution and structural response."""
        NP = self.NP
        n = self.n
        rn = self.rn

        h = (b / 2) / n
        x = np.linspace(0, b / 2 - h, n)
        x1 = np.linspace(h, b / 2, n)

        # Calculate wing loading
        l = np.linspace(0, (b / 2) - (b / 2) / NP, NP)  # noqa: E741
        c1mc4 = c[0] - c[3]
        f_all = np.multiply((3 * b / 10), np.sqrt(np.subtract(1, np.power(x, 2) /
                            np.power(np.divide(b, 2), 2))))
        f1_all = np.multiply((3 * b / 10), np.sqrt(np.subtract(1, np.power(x1, 2) /
                             np.power(np.divide(b, 2), 2))))
        C = c[3] + 2 * ((b / 2 - x) / b) * c1mc4
        C1 = c[3] + 2 * ((b / 2 - x1) / b) * c1mc4
        A_Tot = np.multiply((h / 4) * (C + C1), (np.add(f_all, f1_all)))
        Area = np.sum(A_Tot.reshape((rn, NP), order='F'), axis=0)
        idx1 = np.arange(0, n, rn)
        idx2 = np.arange(rn - 1, n, rn)
        Spanel = (h * rn / 2) * (C[idx1] + C[idx2])

        # Calculate sweep angles
        cosSweep = np.cos(Sweep_40 * np.pi / 180)
        cosInvSweep = 1 / cosSweep
        tanCos2Sweep = np.tan(Sweep_40 * np.pi / 180) * cosSweep * cosSweep

        # Calculate distributed loads
        p = L * Area / np.sum(Area)

        # Calculate shear force and bending moment
        Tcsp = np.cumsum(p)
        Tsp = Tcsp[-1]
        temp = [0] + [Tcsp[i] for i in range(len(Tcsp) - 1)]
        T = np.subtract(Tsp, temp)
        pl = np.multiply(p, l)
        Tcspl = np.cumsum(pl)
        Tspl = Tcspl[-1]
        Mb = np.multiply(np.subtract(np.subtract(Tspl, Tcspl),
                                     np.multiply(l, np.subtract(Tsp, Tcsp))), cosInvSweep)

        # Extract loads at specific points
        idx = np.arange(0, NP, NP // 3)
        P = np.array(T[idx])
        Mz = np.array(Mb[idx])
        Mx = P * D_mx
        # Mz = [Mb[int(i)] for i in np.arange(0, NP - 1, int(NP / 3))]

        # Calculate wing twist due to bending
        I = np.zeros((NP))  # noqa: E741
        chord = c[3] + (np.divide(2 * (b / 2 - l), b)) * c1mc4
        y = np.zeros((2, 9))
        y[0, :] = (l - 0.4 * chord * tanCos2Sweep) * cosInvSweep
        y[1, :] = (l + 0.6 * chord * tanCos2Sweep) * cosInvSweep
        y[1, 0] = 0

        I[0:int(NP / 3)] = np.sqrt((Izz[0]**2 + Izz[1]**2) / 2)
        I[int(NP / 3):int(2 * NP / 3)] = np.sqrt((Izz[1]**2 + Izz[2]**2) / 2)
        I[int(2 * NP / 3):int(NP)] = np.sqrt((Izz[2]**2) / 2)

        La = y[0, 1:NP] - y[0, 0:NP - 1]
        La = np.append(0, La)
        Lb = y[1, 1:NP] - y[1, 0:NP - 1]
        Lb = np.append(0, Lb)

        A = T * La**3 / (3 * E * I) + Mb * La**2 / (2 * E * I)
        B = T * Lb**3 / (3 * E * I) + Mb * Lb**2 / (2 * E * I)
        Slope_A = T * La**2 / (2 * E * I) + Mb * La / (E * I)
        Slope_B = T * Lb**2 / (2 * E * I) + Mb * Lb / (E * I)

        for i in range(NP - 1):
            Slope_A[i + 1] = Slope_A[i] + Slope_A[i + 1]
            Slope_B[i + 1] = Slope_B[i] + Slope_B[i + 1]
            A[i + 1] = A[i] + Slope_A[i] * La[i + 1] + A[i + 1]
            B[i + 1] = B[i] + Slope_B[i] * Lb[i + 1] + B[i + 1]

        bend_twist = ((B - A) / chord) * 180 / np.pi

        # Ensure twist is non-decreasing
        for i in range(1, len(bend_twist)):
            if bend_twist[i] < bend_twist[i - 1]:
                bend_twist[i] = bend_twist[i - 1]

        return P, Mz, Mx, bend_twist, Spanel

    def calculate_structural_response(self):
        """Calculate the complete structural response of the wing."""
        # Calculate wing geometry
        self.c, self.c_box, self.Sweep_40, self.D_mx, self.b, self.a = self.Wing_Mod(self.Z, self.LAMBDA)

        # Calculate moments of inertia
        h = (np.multiply([self.c[i] for i in range(3)], self.beta * float(self.Z[0]))) - \
            np.multiply(0.5, np.add(self.ts1, self.ts3))

        A_top = (np.multiply(self.t1, 0.5 * self.c_box)) + (np.multiply(self.t2, h / 6))
        A_bottom = (np.multiply(self.t3, 0.5 * self.c_box)) + (np.multiply(self.t2, h / 6))
        Y_bar = np.multiply(h, np.divide((2 * A_top), (2 * A_top + 2 * A_bottom)))
        self.Izz = np.multiply(2, np.multiply(A_top, np.power((h - Y_bar), 2))) + \
              np.multiply(2, np.multiply(A_bottom, np.power((-Y_bar), 2)))

        # Calculate loads and deflections
        self.P, self.Mz, self.Mx, self.bend_twist, self.Spanel = self.loads(
            self.b, self.c, self.Sweep_40, self.D_mx, self.L, self.Izz, self.Z, self.E)

        # Calculate torsional deformation
        Phi = (self.Mx / (4 * self.G * (self.c_box * h)**2)) * (self.c_box / self.t1 + 2 * h / self.t2 + self.c_box / self.t3)

        # Calculate total twist
        aa = len(self.bend_twist)
        self.twist = np.array([0] * aa)
        self.twist[0:int(aa/3)] = self.bend_twist[0:int(aa/3)] + Phi[0] * 180 / np.pi
        self.twist[int(aa/3):int(aa*2/3)] = self.bend_twist[int(aa/3):int(aa*2/3)] + Phi[1] * 180 / np.pi
        self.twist[int(aa*2/3):aa] = self.bend_twist[int(aa*2/3):aa] + Phi[2] * 180 / np.pi

        # Calculate total twist contribution
        self.deltaL_divby_q = np.sum(self.twist * self.Spanel * 0.1 * 2)

        # Calculate structural weights
        self.Wtop_alum = (self.b / 4) * (self.c[0] + self.c[3]) * np.mean(self.t1) * self.rho_alum
        self.Wbottom_alum = (self.b / 4) * (self.c[0] + self.c[3]) * np.mean(self.t3) * self.rho_alum
        self.Wside_alum = (self.b / 2) * np.mean(h) * np.mean(self.t2) * self.rho_alum
        self.Wtop_core = (self.b / 4) * (self.c[0] + self.c[3]) * np.mean(np.subtract(self.ts1, self.t1)) * self.rho_core
        self.Wbottom_core = (self.b / 4) * (self.c[0] + self.c[3]) * np.mean(np.subtract(self.ts3, self.t3)) * self.rho_core
        self.Wside_core = (self.b / 2) * np.mean(h) * np.mean(np.subtract(self.ts2, self.t2)) * self.rho_core
        self.W_wingstruct = (self.Wtop_alum + self.Wbottom_alum + self.Wside_alum +
                           self.Wtop_core + self.Wbottom_core + self.Wside_core)
        self.W_fuel_wing = np.mean(h * 0.6 * self.c_box) * (self.b / 3) * 2 * self.rho_fuel

        # Calculate horizontal tail weight
        self.Bh = np.sqrt(self.ARht * self.Sht)
        self.W_ht = 3.316 * ((1 + (self.Fw_at_t / self.Bh))**-2.0) * ((self.L * self.C[2] / 1000)**0.260) * (self.Sht**0.806)

        # Calculate total weights
        Wf = self.C[0] + self.W_fuel_wing
        Ws = self.C[1] + self.W_ht + 2 * self.W_wingstruct
        theta = self.deltaL_divby_q

        return [Ws, Wf, theta]

    def calculate_constraints(self):
        """MATLAB-equivalent implementation of SBJ_constraint_weight"""

        # ---- Inputs (MATCH MATLAB) ----
        # t = np.array(self.t).reshape(-1) / 12.0
        # ts = np.array(self.ts).reshape(-1) / 12.0

        t1 = self.t[0:3]
        t2 = self.t[3:6]
        t3 = self.t[6:9]

        ts1 = self.ts[0:3]
        ts2 = self.ts[3:6]
        ts3 = self.ts[6:9]

        G = np.zeros(72)

        # Constants
        self.E = 10600000 * 144
        self.nu = 0.3
        self.beta = 0.9
        self.k = 6.09375

        # ---- Geometry ----
        self.c, self.c_box, self.Sweep_40, self.D_mx, self.b, self.a = self.Wing_Mod(self.Z, self.LAMBDA)

        c = np.array(self.c)

        l = 0.6 * self.c_box  # IMPORTANT: matches MATLAB

        # ---- Equivalent thickness ----
        teq1 = ((t1**3)/4 + (3*t1)*(ts1 - t1/2)**2)**(1/3)
        teq2 = ((t2**3)/4 + (3*t2)*(ts2 - t2/2)**2)**(1/3)
        teq3 = ((t3**3)/4 + (3*t3)*(ts3 - t3/2)**2)**(1/3)

        # ---- Section geometry ----
        h = self.Z[0] * (self.beta * c[0:3]) - 0.5 * (ts1 + ts3)

        A_top = 0.5 * (t1 * l) + (1/6) * (t2 * h)
        A_bottom = 0.5 * (t3 * l) + (1/6) * (t2 * h)

        Y_bar = h * (2 * A_top) / (2 * A_top + 2 * A_bottom)

        self.Izz = 2 * A_top * (h - Y_bar)**2 + 2 * A_bottom * (-Y_bar)**2

        # ---- Loads ----
        P, Mz, Mx, *_ = self.loads(self.b, c, self.Sweep_40, self.D_mx, self.L, self.Izz, self.Z, self.E)

        # Ensure arrays
        P = np.array(P)
        Mz = np.array(Mz)
        Mx = np.array(Mx)

        # ---- Stresses (FIXED: use Y_bar and element-wise Izz) ----
        sig_1 = Mz * (0.95 * h - Y_bar) / self.Izz
        sig_2 = Mz * (h - Y_bar) / self.Izz
        sig_3 = sig_1
        sig_4 = Mz * (0.05 * h - Y_bar) / self.Izz
        sig_5 = Mz * (-Y_bar) / self.Izz
        sig_6 = sig_4

        q = Mx / (2 * l * h)  # FIXED: uses l

        # ---- Point 1 ----
        T1 = P * (l - self.a) / l  # FIXED
        tau1_T = T1 / (h * t2)
        tau1 = q / t2 + tau1_T

        sig_eq1 = np.sqrt(sig_1**2 + 3 * tau1**2)

        sig_cr1 = (np.pi**2 * self.E * 4 / (12 * (1 - self.nu**2))) * (teq2 / (0.95 * h))**2
        tau_cr1 = (np.pi**2 * self.E * 5.5 / (12 * (1 - self.nu**2))) * (teq2 / (0.95 * h))**2

        G[0:3] = self.k * sig_eq1
        G[3:6] = self.k * ((sig_1 / sig_cr1) + (tau1 / tau_cr1)**2)
        G[6:9] = self.k * ((-sig_1 / sig_cr1) + (tau1 / tau_cr1)**2)

        # ---- Point 2 ----
        tau2 = q / t1
        sig_eq2 = np.sqrt(sig_2**2 + 3 * tau2**2)

        sig_cr2 = (np.pi**2 * self.E * 4 / (12 * (1 - self.nu**2))) * (teq1 / l)**2
        tau_cr2 = (np.pi**2 * self.E * 5.5 / (12 * (1 - self.nu**2))) * (teq1 / l)**2

        G[9:12] = self.k * sig_eq2
        G[12:15] = self.k * ((sig_2 / sig_cr2) + (tau2 / tau_cr2)**2)
        G[15:18] = self.k * ((-sig_2 / sig_cr2) + (tau2 / tau_cr2)**2)

        # ---- Point 3 ----
        T2 = P * self.a / l  # FIXED
        tau3_T = -T2 / (h * t2)
        tau3 = q / t2 + tau3_T

        sig_eq3 = np.sqrt(sig_3**2 + 3 * tau3**2)

        sig_cr3 = sig_cr1
        tau_cr3 = tau_cr1

        G[18:21] = self.k * sig_eq3
        G[21:24] = self.k * ((sig_3 / sig_cr3) + (tau3 / tau_cr3)**2)
        G[24:27] = self.k * ((-sig_3 / sig_cr3) + (tau3 / tau_cr3)**2)

        # ---- Point 4 ----
        tau4 = -q / t2 + tau1_T
        sig_eq4 = np.sqrt(sig_4**2 + 3 * tau4**2)

        G[27:30] = self.k * sig_eq4

        # ---- Point 5 ----
        tau5 = q / t3
        sig_eq5 = np.sqrt(sig_5**2 + 3 * tau5**2)

        sig_cr5 = (np.pi**2 * self.E * 4 / (12 * (1 - self.nu**2))) * (teq3 / l)**2
        tau_cr5 = (np.pi**2 * self.E * 5.5 / (12 * (1 - self.nu**2))) * (teq3 / l)**2

        G[30:33] = self.k * sig_eq5
        G[33:36] = self.k * ((sig_5 / sig_cr5) + (tau5 / tau_cr5)**2)
        G[36:39] = self.k * ((-sig_5 / sig_cr5) + (tau5 / tau_cr5)**2)

        # ---- Point 6 ----
        tau6 = -q / t2 + tau3_T
        sig_eq6 = np.sqrt(sig_6**2 + 3 * tau6**2)

        G[39:42] = self.k * sig_eq6

        # ---- Constraints ----
        Sig_C = 65000 * 144
        Sig_T = 65000 * 144

        G1 = np.zeros(72)

        G1[0:3] = (G[0:3] / Sig_C) - 1
        G1[54:57] = -(G[0:3] / Sig_C) - 1
        G1[3:9] = G[3:9] - 1

        G1[9:12] = (G[9:12] / Sig_C) - 1
        G1[57:60] = -(G[9:12] / Sig_C) - 1
        G1[12:18] = G[12:18] - 1

        G1[18:21] = (G[18:21] / Sig_C) - 1
        G1[60:63] = -(G[18:21] / Sig_C) - 1
        G1[21:27] = G[21:27] - 1

        G1[27:30] = (G[27:30] / Sig_T) - 1
        G1[63:66] = -(G[27:30] / Sig_T) - 1

        G1[30:33] = (G[30:33] / Sig_T) - 1
        G1[66:69] = -(G[30:33] / Sig_T) - 1
        G1[33:39] = G[33:39] - 1

        G1[39:42] = (G[39:42] / Sig_T) - 1
        G1[69:72] = -(G[39:42] / Sig_T) - 1

        # ---- Thickness constraints ----
        G1[42:45] = 0.5 * (ts1 + ts3) / h - 1
        G1[45:48] = t1 / (ts1 - 0.1 * t1) - 1
        G1[48:51] = t2 / (ts2 - 0.1 * t2) - 1
        G1[51:54] = t3 / (ts3 - 0.1 * t3) - 1

        return G1.tolist()

    def SBJ_wing_structural_design(self):
        """Run the complete wing design analysis."""
        # Calculate structural response
        Ws, Wf, theta = self.calculate_structural_response()

        # Calculate constraints
        G1 = self.calculate_constraints()

        # Return results
        return {
            'Lift': self.Lift,
            'Ws': Ws,
            'Wf': Wf,
            'theta': theta,
            'G1': G1,
            'twist': self.twist,
            'Spanel': self.Spanel,
            'bend_twist': self.bend_twist
        }

    def SBJ_structure_analysis(self):
        return self.calculate_structural_response()

    def SBJ_structure_opt(self):
        return [0, self.calculate_constraints()]

    def print_results(self):
        """Print the analysis results."""
        out_dict = self.SBJ_wing_structural_design()
        print("=== Wing Design Analysis Results ===")
        print(f"Lift = {out_dict['Lift']}")
        print(f"Theta = {out_dict['theta']}")
        print(f"Ws = {out_dict['Ws']}")
        print(f"Wf = {out_dict['Wf']}")
        print(f"Twist = {out_dict['twist']}")
        print(f"Spanel = {out_dict['Spanel']}")
        print(f"Bend Twist = {out_dict['bend_twist']}")
        print(f"Constraints (G1) = {out_dict['G1']}")
        CV = []
        for i, g in enumerate(out_dict['G1']):
          if g > 0:
              CV.append((i, g))
        print(f"Constraints Violation (G1) = {CV}")
        print("==================================")

if __name__ == "__main__":
    # twist =  [0 0 0 0 0 0 0 0 0]
    # Spanel =  [41.23931624 37.91547958 34.59164292 31.26780627 27.94396961 24.62013295
    #  21.2962963  17.97245964 14.64862298]
    # t =  [0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25]
    # ts =  [0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5]
    # Lift =  25000
    # Theta =  0.0
    # Ws = 20293.722985582557
    # Wf = 2158.127657838667
    wda = WingDesignAnalyzer()
    wda.print_results()

=== Wing Design Analysis Results ===
Lift = 34300
Theta = 0.0
Ws = 17534.478329721256
Wf = 10285.191037913619
Twist = [0 0 0 0 0 0 0 0 0]
Spanel = [64.34191919 57.60454545 50.86717172 44.12979798 37.39242424 30.65505051
 23.91767677 17.18030303 10.44292929]
Bend Twist = [0.         0.15623242 0.20883762 0.24842373 0.26490628 0.26490628
 0.26490628 0.26490628 0.26490628]
Constraints (G1) = [-0.7971430201325933, -0.8271875804774498, -0.8694219504053378, -0.9946429154494736, -0.9972909777404295, -0.9894677785723329, -1.005357040918779, -1.002709015560648, -1.0105317358777022, -0.7750126093089859, -0.8057620279249424, -0.8551800278937667, 0.0004591669530926712, -0.013142623776765783, -0.0022847305162341236, -1.9991547592894157, -1.984639568158407, -1.993602496858708, -0.7951371082154435, -0.8261490159278695, -0.8659048683279565, -0.9946428985139183, -0.9972909751402593, -0.9894675901073092, -1.005357023983224, -1.0027090129604777, -1.0105315474126784, -0.7939284435175177, -0.82552131081210

In [ ]:
#@title #####*Structural Wing Subproblem Test*


def SBJ_structure_opt_test(x):
  ssbj_structure = WingDesignAnalyzer(
      Lift=x[0],
      tc=x[1],
      ARw=x[2],
      LAMBDAw=x[3],
      Sref=x[4],
      Sht=x[5],
      ARht=x[6],
      lambdatr=x[7],
      t=x[8: 17],
      ts=x[17:])
  return ssbj_structure.SBJ_structure_opt()

def test_SBJ_structural():

  # Rest of your code follows
  is_mac = platform.platform().split('-')[0] == 'macOS'
  lb = [5000.0,
        0.01,
        2.5,
        40.0,
        200.0,
        50.0,
        2.5,
        0.1,
        0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1,
        0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1,
        ]
  ub = [100000.0,
        0.1,
        8.0,
        70.0,
        800.0,
        148.9,
        8.5,
        0.4,
        4., 4., 4., 4., 4., 4., 4., 4., 4.,
        9., 9., 9., 9., 9., 9., 9., 9., 9.,
        ]
  bl = [25000.0,
        0.050,
        3.0,
        60.0,
        500.0,
        100.0,
        5.5,
        0.3,
        3., 3., 3., 3., 3., 3., 3., 3., 3.,
        6., 6., 6., 6., 6., 6., 6., 6., 6.,
        ]
  vnames = ["L", "tc", "ARw", "LAMBDAw", "Sref", "Sht", "ARht", "lambdatr",
            "t", "t", "t", "t", "t", "t", "t", "t", "t",
            "tc", "tc", "tc", "tc", "tc", "tc", "tc", "tc", "tc"
            ]
  vartype = ["REAL"] * 26
  constants = None
  constants_name = None
  wd = "/content/sample_data"

  param = {"baseline": bl,
           "lb": lb,
           "ub": ub,
           "var_names": vnames,
           "var_type": vartype,
           "var_sets": None,
           "scaling": [1] * len(lb),
           # "post_dir": "./post",
           "constants": constants,
           "constants_name": constants_name,
           "name": "SBJ_Struct",
           "mesh_type": "GMESH",
           "post_dir": os.path.join(wd, "test_SBJ_post"),
           "constraints_type": ["PB"] * 72,
           "lambda_multipliers": [1000] * 72,
           "rho": 0.01,
           "h_max": np.inf,

           }

  options = {
      "seed": 10000,
      "budget": 500,
      "tol": 1E-12,
      "psize_init": 1,
      "display": False,
      "opportunistic": False,
      "check_cache": True,
      "store_cache": True,
      "collect_y": False,
      "rich_direction": True,
      "precision": "high" if is_mac else "medium",
      "save_results": False,
      "save_coordinates": False,
      "save_all_best": False,
      "parallel_mode": False
  }
  # isWin = platform.platform().split('-')[0] == 'Windows'
  # options["precision"] = "high" if isWin else "medium"

  search_dict = {
      "type": "sampling",
              "s_method": "ACTIVE",
              "ns": 10,
              "visualize": False,
              "criterion": None
  }

  data = {"evaluator": {"blackbox": SBJ_structure_opt_test}, "param": param,
          "options": options, "search": search_dict}

  h = 55000
  M = 1.4

  out_mads, _, _ = mads.main(data)
  print("=== Testing Structural Wing Subproblem: Minimum Candidate")
  print(f"xmin={out_mads["xmin"]}")
  print(f"fmin_feasible={out_mads["fmin"]}")
  print(f"hmin_feasible={out_mads["hmin"]}")




test_SBJ_structural()

SBJ_Struct: |Pending:  █  , Feasible:  █  , Infeasible:  █  |█████████████████████████████████████████████████| 105/500 #Evaluated
SBJ_Struct: |Pending:  █  , Feasible:  █  , Infeasible:  █  |█████████████████████████████████████████████████| 189/500 #Evaluated
SBJ_Struct: |Pending:  █  , Feasible:  █  , Infeasible:  █  |████████████████████████████████████████████████| 273/500 #Evaluated
SBJ_Struct: |Pending:  █  , Feasible:  █  , Infeasible:  █  |█████████████████████████████████████████████████| 362/500 #Evaluated
SBJ_Struct: |Pending:  █  , Feasible:  █  , Infeasible:  █  |████████████████████████████████████████████████| 451/500 #Evaluated
SBJ_Struct: |Pending:  █  , Feasible:  █  , Infeasible:  █  |█████████████████████████████████████████████████| 500/500 #Evaluated
=== Testing Structural Wing Subproblem: Minimum Candidate
xmin=[100000.0, 0.1, 3.0, 70.0, 800.0, 100.0, 5.5, 0.4, 3.0, 3.0, 4.0, 4.0, 4.0, 3.0, 4.0, 3.0, 4.0, 6.0, 9.0, 6.0, 9.0, 6.0, 9.0, 6.0, 6.0, 9.0]
fmin_feasibl

##### *Aerodynamic Analysis*

The Aerodynamics subproblem in the SBJ distributed MDO benchmark is responsible for simulating the aircraft's lift and drag characteristics under cruise conditions. It is one of the most influential disciplines because it directly affects both force balance (lift = weight, thrust = drag) and mission performance (range via lift-to-drag ratio).


The aerodynamics module acts as:

* A performance predictor → computes lift L and drag D
* A coupling driver → strongly influences both propulsion (through drag) and structures (through lift-induced loads)
* A constraint contributor → supports lift–weight consistency

It receives geometry and flight conditions, then returns aerodynamic coefficients.

The aerodynamics discipline is given below:

\begin{equation*}
	\begin{aligned}
		& \underset{\mathbf{x}}{\text{minimize}}
		& & \phi\left(W_{\mathrm{f}}^{\mathrm{a}}-W_{\mathrm{f}}^{\mathrm{s}}\right)+\phi\left(W_{\mathrm{s}}^{\mathrm{a}}-W_{\mathrm{s}}^{\mathrm{s}}\right)+\phi\left(L^{\mathrm{s}}-L^{\mathrm{ae}}\right)+\phi\left(\theta^{\mathrm{ae}}-\theta^{\mathrm{s}}\right) \\
    & & & +\phi\left(t / c^{\mathrm{ae}}-t / c^{\mathrm{s}}\right)+\phi\left(\mathrm{AR}_{\mathrm{w}}^{\mathrm{ae}}-\mathrm{AR}_{\mathrm{w}}^{\mathrm{s}}\right) \\
    & & & +\phi\left(\Lambda_{\mathrm{w}}^{\mathrm{ae}}-\Lambda_{\mathrm{w}}^{\mathrm{s}}\right)+\phi\left(S_{\mathrm{ref}}^{\mathrm{ae}}-S_{\mathrm{ref}}^{\mathrm{s}}\right)+\phi\left(S_{\mathrm{ht}}^{\mathrm{ae}}-S_{\mathrm{ht}}^{\mathrm{s}}\right)+\phi\left(\mathrm{AR}_{\mathrm{ht}}^{\mathrm{ae}}-\mathrm{AR}_{\mathrm{ht}}^{\mathrm{s}}\right) \\
    & \text{subject to}
		& & \mathbf{g}_{\text {struc }}\left(L^{\mathrm{s}}, t / c^{\mathrm{s}}, \mathrm{AR}_{\mathrm{w}}^{\mathrm{s}}, \Lambda_{\mathrm{w}}^{\mathrm{s}}, S_{\mathrm{ref}}^{\mathrm{s}}, S_{\mathrm{ht}}^{\mathrm{s}}, \mathrm{AR}_{\mathrm{ht}}^{\mathrm{s}},[t],\left[t_{\mathrm{s}}\right], \lambda\right) \leq \mathbf{0} \\
    & \text{while solving}
		& & W_{\mathrm{s}}^{\mathrm{s}}=W_{\mathrm{s}}\left(L^{\mathrm{s}}, t / c^{\mathrm{s}}, \mathrm{AR}_{\mathrm{w}}^{\mathrm{s}}, \Lambda_{\mathrm{w}}^{\mathrm{s}}, S_{\mathrm{ref}}^{\mathrm{s}},,_{\mathrm{ht}}^{\mathrm{s}}, \mathrm{AR}_{\mathrm{ht}}^{\mathrm{s}},[t],\left[t_{\mathrm{s}}\right], \lambda\right) \\
    & & & W_{\mathrm{f}}^{\mathrm{s}}=W_{\mathrm{f}}\left(L^{\mathrm{s}}, t / c^{\mathrm{s}}, \mathrm{AR}_{\mathrm{w}}^{\mathrm{s}}, \Lambda_{\mathrm{w}}^{\mathrm{s}}, S_{\mathrm{ref}}^{\mathrm{s}}, S_{\mathrm{ht}}^{\mathrm{s}}, \mathrm{AR}_{\mathrm{ht}}^{\mathrm{s}},[t],\left[t_{\mathrm{s}}\right], \lambda\right) \\
    & & & \theta^{\mathrm{s}}=\theta\left(L^{\mathrm{s}}, t / c^{\mathrm{s}}, \mathrm{AR}_{\mathrm{w}}^{\mathrm{s}}, \Lambda_{\mathrm{w}}^{\mathrm{s}}, S_{\mathrm{ref}}^{\mathrm{s}}, S_{\mathrm{ht}}^{\mathrm{s}}, \mathrm{AR}_{\mathrm{ht}}^{\mathrm{s}},[t],\left[t_{\mathrm{s}}\right], \lambda\right)\\
    & \text{where}
		& & \mathbf{x} = \left[L^{\mathrm{s}}, t / c^{\mathrm{s}}, \mathrm{AR}_{\mathrm{w}}^{\mathrm{s}}, \Lambda_{\mathrm{w}}^{\mathrm{s}}, S_{\mathrm{ref}}^{\mathrm{s}}, S_{\mathrm{ht}}^{\mathrm{s}}, \mathrm{AR}_{\mathrm{ht}}^{\mathrm{s}},[t],\left[t_{\mathrm{s}}\right], \lambda\right]^\textit{T}
	\end{aligned}
\end{equation*}

There is no local objective in this subproblem. The calculation of the coupling variables $W_{\mathrm{s}}^{\mathrm{s}}$, $W_{\mathrm{f}}^{\mathrm{s}}$, and $\theta^{\mathrm{s}}$, is given by `SBJ_aerodynamics_analysis`. The calculation of the constraints $\mathbf{g}_{\mathrm{struc}}$ and definition of the optimization problem is given by `SBJ_aerodynamics_opt`.

In [ ]:
#@title #####*Aerodynamic Class: Analysis and Subproblem Functions*
import numpy as np
import copy

class SSBJAerodynamics:
    """
    Class for SSBJ (Supersonic Business Jet) aerodynamic calculations.
    This class implements the drag polar and constraint calculations
    for supersonic aircraft design.
    """

    def __init__(self, h=55000, Mach=1.4, tc=0.05, ARw=3.0, LAMBDAw=60.0, LAMBDAht=45.0,
                 Sref=500.0, Sht=100.0, ARht=5.5, Lw=0.15, Lht=1.5, Wt=25000, theta=10.0, ESFp=1.0):
        """
        Initialize the SSBJ aerodynamics class with given parameters.

        Args:
            h: Altitude in feet
            Mach: Mach number
            tc: Thickness-to-chord ratio
            ARw: Wing aspect ratio
            LAMBDAw: Wing sweep angle in degrees
            LAMBDAht: Horizontal tail sweep angle in degrees
            Sref: Wing reference area in ft²
            Sht: Horizontal tail area in ft²
            ARht: Horizontal tail aspect ratio
            Lw: Wing moment arm
            Lht: Horizontal tail moment arm
            Wt: Total aircraft weight in lb
            theta: Incidence angle in degrees
            ESFp: Equivalent skin friction coefficient
        """
        # Constants
        self.C = [500.0, 16000.0, 4.0, 4360.0, 0.01375, 1.0]  # CDminM = C[4]
        self.Nh = self.C[5]

        # Input parameters
        self.h = h
        self.Mach = Mach
        self.tc = tc
        self.ARw = ARw
        self.LAMBDAw = LAMBDAw
        self.LAMBDAht = LAMBDAht
        self.Sref = Sref
        self.Sht = Sht
        self.ARht = ARht
        self.Lw = Lw
        self.Lht = Lht
        self.Wt = Wt
        self.theta = theta
        self.ESFp = ESFp

        # Local variables
        self.Z = [self.tc, self.h, self.Mach, self.ARw, self.LAMBDAw, self.Sref, self.Sht, self.ARht]

        # Initialize results
        self.Lift = 0
        self.Drag = 0
        self.LDr = 0
        self.Pg = 0
        self.g1 = 0
        self.g2 = 0
        self.g3 = 0
        self.CLo = np.array([0.0, 0.0])
        self.DCL = np.array([0.0, 0.0])

    def poly_approx(self, S, S_new, flag, S_bound):
        """
        Polynomial approximation function for scaling factors.

        Args:
            S: Base values
            S_new: New values to scale
            flag: Flag for different calculation modes
            S_bound: Boundaries for scaling

        Returns:
            FF: Scaled factor
        """
        S_norm = []
        S_shifted = []
        Ai = []
        Aij = np.zeros((len(S), len(S)))

        for i in range(len(S)):
            S_norm.append(S_new[i] / S[i])
            if S_norm[i] > 1.25:
                S_norm[i] = 1.25
            elif S_norm[i] < 0.75:
                S_norm[i] = 0.75
            S_shifted.append(S_norm[i] - 1)
            a = 0.1
            b = a

            if flag[i] == 5:
                # CALCULATE POLYNOMIAL COEFFICIENTS (S-ABOUT ORIGIN)
                So = 0
                Sl = So - S_bound[i]
                Su = So + S_bound[i]
                Mtx_shifted = np.array([[1, Sl, Sl**2], [1, So, So**2], [1, Su, Su**2]])

                F_bound = np.array([1 + (.5*a)**2, 1, 1 + (.5*b)**2])
                A = np.linalg.solve(Mtx_shifted, F_bound)
                Ao = A[0]
                Ai.append(A[1])
                Aij[i, i] = A[2]

                # CALCULATE POLYNOMIAL COEFFICIENTS
            else:
                if flag[i] == 0:
                    S_shifted.append(0)
                elif flag[i] == 3:
                    a *= -1
                    b = copy.deepcopy(a)
                elif flag[i] == 2:
                    b = 2 * a
                elif flag[i] == 4:
                    a *= -1
                    b = 2*a
                # DETERMINE BOUNDS ON FF DEPENDING ON SLOPE-SHAPE
                # CALCULATE POLYNOMIAL COEFFICIENTS (S-ABOUT ORIGIN)
                So = 0
                Sl = So - S_bound[i]
                Su = So + S_bound[i]
                Mtx_shifted = np.array([[1, Sl, Sl**2], [1, So, So**2], [1, Su, Su**2]])
                F_bound = np.array([1 - .5*a, 1, 1 + .5*b])
                A = np.linalg.solve(Mtx_shifted, F_bound)
                Ao = A[0]
                Ai.append(A[1])
                Aij[i, i] = A[2]

                # CALCULATE POLYNOMIAL COEFFICIENTS

        # Correlation matrix
        R = np.array([[0.2736, 0.3970, 0.8152, 0.9230, 0.1108],
                      [0.4252, 0.4415, 0.6357, 0.7435, 0.1138],
                      [0.0329, 0.8856, 0.8390, 0.3657, 0.0019],
                      [0.0878, 0.7248, 0.1978, 0.0200, 0.0169],
                      [0.8955, 0.4568, 0.8075, 0.9239, 0.2525]])

        for i in range(len(S)):
            for j in range(i+1, len(S)):
                Aij[i, j] = Aij[i, i] * R[i, j]
                Aij[j, i] = Aij[i, j]

        S_shifted = np.array(S_shifted)
        FF = Ao + np.dot(Ai, np.transpose(S_shifted)) + (1/2) * np.dot(np.dot(S_shifted, Aij), np.transpose(S_shifted))
        return FF

    def calculate_drag_polar(self):
        """
        Calculate the drag polar and related parameters.
        """
        # Extract variables from Z
        ARht = self.Z[7]
        S_ht = self.Z[6]

        # Calculate velocity and density
        if self.Z[1] < 36089:
            V = self.Z[2] * (1116.39 * np.sqrt(1 - (6.875e-06 * self.Z[1])))
            rho = (2.377e-03) * (1 - (6.875e-06 * self.Z[1]))**4.2561
        else:
            V = self.Z[2] * 968.1
            rho = (2.377e-03) * (.2971) * np.exp(-(self.Z[1] - 36089) / 20806.7)

        q = 0.5 * rho * (V**2)

        # Scale coefficients for proper conditioning of matrix A
        a = q * self.Z[5] / 1e5
        b = self.Nh * q * S_ht / 1e5
        c = self.Lw
        d = self.Lht * self.Nh * (S_ht / self.Z[5])

        A = np.array([[a, b], [c, d]])

        # Scale coefficient Wt for proper conditioning of matrix A
        B = np.array([self.Wt / 1e5, 0])

        # Solve for CLo
        try:
            CLo = np.linalg.solve(A, B)
        except:  # noqa: E722
            CLo = np.array([-np.inf, np.inf])

        # Calculate delta_L
        delta_L = self.theta * q
        Lw1 = CLo[0] * q * self.Z[5] - delta_L
        CLw1 = Lw1 / (q * self.Z[5])
        CLht1 = -CLw1 * c / d

        # Scale first coefficient of D for proper conditioning of matrix A
        D = np.array([(self.Wt - CLw1 * a - CLht1 * b) / 1e5, -CLw1 * c - CLht1 * d])

        # Solve for DCL
        try:
            self.DCL = np.linalg.solve(A, D)
        except:  # noqa: E722
            self.DCL = np.array([np.nan, np.nan])

        # Calculate induced drag factors
        if self.Z[2] >= 1:
            kw = self.Z[3] * (self.Z[2]**2 - 1) * np.cos(self.Z[4] * np.pi / 180) / \
            (4 * self.Z[3] * np.sqrt(self.Z[2]**2 - 1) - 2)
            kht = ARht * (self.Z[2]**2 - 1) * np.cos(self.LAMBDAht * np.pi / 180) / \
            (4 * ARht * np.sqrt(self.Z[2]**2 - 1) - 2)
        else:
            kw = 1 / (np.pi * 0.8 * self.Z[3])
            kht = 1 / (np.pi * 0.8 * ARht)

        # Calculate Fo1
        S_initial1 = copy.deepcopy(self.ESFp)
        S1 = copy.deepcopy(self.ESFp)
        flag1 = 1
        bound1 = 0.25
        Fo1 = self.poly_approx(S_initial1 if isinstance(S_initial1, list) else [S_initial1],
                              S1 if isinstance(S1, list) else [S1],
                              flag1 if isinstance(flag1, list) else [flag1],
                              bound1 if isinstance(bound1, list) else [bound1])

        # Calculate minimum drag coefficient
        CDmin = self.C[4] * Fo1 + 3.05 * (self.Z[0]**(5/3)) * ((np.cos(self.Z[4] * np.pi / 180))**(3/2))

        # Calculate total drag coefficients
        CDw = CDmin + kw * (CLo[0]**2) + kw * (self.DCL[0]**2)
        CDht = kht * (CLo[1]**2) + kht * (self.DCL[1]**2)
        CDp = CDw + CDht
        CLp = CLo[0] + CLo[1]

        # Calculate lift and drag
        Lift = self.Wt
        Drag = q * CDw * self.Z[5] + q * CDht * self.Z[6]
        LDr = CLp / CDp

        # Calculate adverse pressure gradient (G2)
        S_initial2 = copy.deepcopy(self.tc)
        S2 = copy.deepcopy(self.Z[0])
        flag1 = [1]
        bound1 = [0.25]

        Pg = self.poly_approx(S_initial2 if isinstance(S_initial2, list) else [S_initial2],
                             S2 if isinstance(S2, list) else [S2],
                             flag1 if isinstance(flag1, list) else [flag1],
                             bound1 if isinstance(bound1, list) else [bound1])

        return [Lift, Drag, LDr, Pg, CLo[0], CLo[1]]

    def calculate_constraints(self, Pg, CLo):
        """
        Calculate the constraints for the aerodynamic analysis.
        """

        # Constraints
        Pg_uA = 1.1
        if CLo[0] > 0:
            g2 = (2 * CLo[1]) - CLo[0]
            g3 = (2 * (-CLo[1])) - CLo[0]
        else:
            g2 = (2 * (-CLo[1])) - CLo[0]
            g3 = (2 * CLo[1]) - CLo[0]

        g1 = Pg / Pg_uA - 1

        return [g1, g2, g3]

    def SBJ_aerodynamics_analysis(self):
        """
        Run the complete aerodynamic analysis.
        """
        return self.calculate_drag_polar()

    def SBJ_aerodynamics_opt(self, Pg, CLo):
        """
        Run the complete aerodynamic analysis.
        """

        return [0, self.calculate_constraints(Pg, CLo)]

    def print_results(self):
        """Print the analysis results."""
        Lift, Drag, LDr, Pg, CLo0, CLo1 = self.SBJ_aerodynamics_analysis()
        g1, g2, g3 = self.calculate_constraints(Pg, [CLo0, CLo1])
        print("=== Wing Aerodynamic Analysis Results ===")
        print(f"Lift = {Lift}")
        print(f"Drag = {Drag}")
        print(f"LDr = {LDr}")
        print(f"Pg = {Pg}")
        print(f"Clo0 = {CLo0}")
        print(f"CLo1 = {CLo1}")
        print(f"g1 = {g1}")
        print(f"g2 = {g2}")
        print(f"g3 = {g3}")
        print("==================================")

# Example usage:
if __name__ == "__main__":
    # Create an instance with default parameters
    aerodynamics = SSBJAerodynamics()
    # Run the analysis
    print(aerodynamics.print_results())

# /SSBJ_Aerodynamics.py
# Drag =  4608.661215792792
# LDr =  2.7450900636447657
# Lift =  25000
# G2 =  1.0
# g1 =  -0.09090909090909094
# g2 =  -0.42509404401935696
# g3 =  -5.551115123125783e-17

=== Wing Aerodynamic Analysis Results ===
Lift = 25000
Drag = 4608.661215792792
LDr = 2.7450900636447657
Pg = 1.0
Clo0 = 0.2125470220096785
CLo1 = -0.10627351100483924
g1 = -0.09090909090909094
g2 = -0.425094044019357
g3 = -2.7755575615628914e-17
None


# MDO Run

The following `Python` code is an implementation to the decomposed optimization subproblems stated in [1].

In [ ]:
import os
import logging # Added this line

from DMDO import MDA, MDO, USER, main
import pandas # Import pandas to access its options

import warnings
warnings.filterwarnings("ignore")



# Disable numexpr for pandas computations to avoid compatibility issues
pandas.options.compute.use_numexpr = False

# Configure basic logging to ensure a default logger is available
logging.basicConfig(level=logging.INFO) # Added this line
user = USER

user.h = 55000
user.M = 1.4

def SBJ_aircraft_analysis(x):
  ssbj_aircraft = SSBJ_Aircraft(We=x[1], Ws=x[3], Wf=x[4], SFCp=x[0], LDr=x[2])
  return ssbj_aircraft.SBJ_aircraft_analysis()

def SBJ_propulsion_analysis(x):
  ssbj_prop = SSBJPropulsion(D=x[0], T=x[1])
  return ssbj_prop.SBJ_propulsion_analysis()

def SBJ_aerodynamics_analysis(x):
  ssbj_prop = SSBJAerodynamics(Wt=x[0], ESFp=x[1], theta=x[2], tc=x[3], ARw=x[4], LAMBDAw=x[5], Sref=x[6], Sht=x[7], \
                               ARht=x[8], LAMBDAht=x[9], Lw=x[10], Lht=x[11])
  return ssbj_prop.SBJ_aerodynamics_analysis()

def SBJ_structure_analysis(x):
  ssbj_structure = WingDesignAnalyzer(Lift=x[0], tc=x[1],ARw=x[2], LAMBDAw=x[3], Sref=x[4], Sht=x[5], ARht=x[6], \
                                      lambdatr=x[7], t=x[8:17], ts=x[17:])
  return ssbj_structure.SBJ_structure_analysis()

def SBJ_aircraft_opt(x, y, *args):
  x = np.array(x, dtype=np.float64)
  y = np.array(y, dtype=np.float64)
  ssbj_aircraft = SSBJ_Aircraft(We=x[1], Ws=x[3], Wf=x[4], SFCp=x[0], LDr=x[2])
  return ssbj_aircraft.SBJ_aircraft_opt(Wt=float(y[0]), range=float(y[1]))

def SBJ_propulsion_opt(x, y, *args):
  x = np.array(x, dtype=np.float64)
  y = np.array(y, dtype=np.float64)
  ssbj_prop = SSBJPropulsion(D=x[0], T=x[1])
  return ssbj_prop.SBJ_propulsion_opt(Temp_E=float(y[3]), Throttle_uA=float(y[4]))

def SBJ_aerodynamics_opt(x, y, *args):
  x = np.array(x, dtype=np.float64)
  y = np.array(y, dtype=np.float64)
  ssbj_prop = SSBJAerodynamics(Wt=x[0], ESFp=x[1], theta=x[2], tc=x[3], ARw=x[4], LAMBDAw=x[5], Sref=x[6], Sht=x[7], \
                               ARht=x[8], LAMBDAht=x[9], Lw=x[10], Lht=x[11]) # Fix: y[9] was passed as x[9], which would be incorrect.
  return ssbj_prop.SBJ_aerodynamics_opt(Pg=float(y[3]), CLo=[float(y[4]), float(y[5])])

def SBJ_structure_opt(x, y, *args):
  x = np.array(x, dtype=np.float64)
  y = np.array(y, dtype=np.float64)
  ssbj_structure = WingDesignAnalyzer(Lift=x[0], tc=x[1],ARw=x[2], LAMBDAw=x[3], Sref=x[4], Sht=x[5], ARht=x[6], \
                                      lambdatr=x[7], t=x[8:17], ts=x[17:])
  return ssbj_structure.SBJ_structure_opt()

def SBJ_auto_build_and_run(exec_mode: str = "parallel"):
  csd = os.path.dirname(os.path.abspath(f"/content/sample_data/SBJ_{exec_mode}/SBJ.yaml"))
  print(csd)
  p_file: str = os.path.join(csd, "SBJ.yaml")

  def_dict = {
    "setup_file": p_file,
    "run_mode": "build",
    # "is_resume": 0,
    # "restart_file": None,
    "exec_mode": exec_mode,
    "mdo_name": "SBJ",
    "working_dir": csd
  }
  MDAO: MDO = main(def_dict)
  for i in range(len(MDAO.subProblems)):
    temp :MDA = MDAO.subProblems[i].MDA_process
    for j in range(len(temp.analyses)):
      MDAO.subProblems[i].MDA_process.analyses[j].blackbox = globals()[MDAO.subProblems[i].MDA_process.analyses[j].blackbox]
    MDAO.subProblems[i].opt = globals()[MDAO.subProblems[i].opt]
  out = MDAO.run(os.path.join(csd, "SBJ.out"), mode = exec_mode)
  print(out)

SBJ_auto_build_and_run(exec_mode=exec_modes[0])
SBJ_auto_build_and_run(exec_mode=exec_modes[1])

/content/sample_data/SBJ_serial
SP_1: |Pending:  █  , Feasible:  █  , Infeasible:  █  |██████████████████████████████████████████████████| 11/50 #Evaluated
SP_1: |Pending:  █  , Feasible:  █  , Infeasible:  █  |██████████████████████████████████████████████████| 22/50 #Evaluated
SP_1: |Pending:  █  , Feasible:  █  , Infeasible:  █  |██████████████████████████████████████████████████| 33/50 #Evaluated
SP_1: |Pending:  █  , Feasible:  █  , Infeasible:  █  |██████████████████████████████████████████████████| 44/50 #Evaluated
SP_1: |Pending:  █  , Feasible:  █  , Infeasible:  █  |██████████████████████████████████████████████████| 48/50 #Evaluated
SP_1: |Pending:  █  , Feasible:  █  , Infeasible:  █  |██████████████████████████████████████████████████| 50/50 #Evaluated
SP_2: |Pending:  █  , Feasible:  █  , Infeasible:  █  |██████████████████████████████████████████████████| 7/50 #Evaluated
SP_2: |Pending:  █  , Feasible:  █  , Infeasible:  █  |██████████████████████████████████████████████

# SBJ MDO Results

### Postprocess Coordination Results

Before creating dynamic plots, let's examine the structure of the `Coordination_history.out` file to understand how to parse the data correctly.

In [ ]:
import pandas as pd
import plotly.express as px
import os
import io


coord_history_path_1 = os.path.join(f'/content/sample_data/SBJ_{exec_modes[0]}', 'Coordination_history.out')
coord_history_path_2 = os.path.join(f'/content/sample_data/SBJ_{exec_modes[1]}', 'Coordination_history.out')


# Define column names based on the user's input
column_names = [
    'Time', 'Iteration #', 'Max. inconsistency', 'Objective', 'Status',
    'Variables change', 'Maximum penalty', 'Coupling_with_qmax', 'SFC_1',
    'We_1', 'Wt_1', 'LD_1', 'Ws_1', 'Wf_1', 'SFC_2', 'We_2', 'D_2', 'ESF_2',
    'T_2', 'D_3', 'ESF_3', 'Wt_3', 'LD_3', 'theta_3', 'L_3', 'tc_3', 'ARw_3',
    'LAMBDAw_3', 'Sref_3', 'Sht_3', 'ARht_3', 'LAMBDAht_3', 'Lw_3', 'Lht_3',
    'theta_4', 'L_4', 'Ws_4', 'Wf_4', 'tc_4', 'ARw_4', 'LAMBDAw_4', 'Sref_4',
    'Sht_4', 'ARht_4', 'lambda_4', 't_4_1', 't_4_2', 't_4_3', 't_4_4', 't_4_5',
    't_4_6', 't_4_7', 't_4_8', 't_4_9', 'ts_4_1', 'ts_4_2', 'ts_4_3', 'ts_4_4',
    'ts_4_5', 'ts_4_6', 'ts_4_7', 'ts_4_8', 'ts_4_9', 'range_1', 'Temp_E_2',
    'Throttle_uA_2', 'Pg_3', 'CLo1_3', 'CLo2_3'
]

if os.path.exists(coord_history_path_1):
    with open(coord_history_path_1, 'r') as f:
        file_content = f.read()
        df_coord_results_1 = pd.read_csv(io.StringIO(file_content), sep=',\s*', engine='python', names=column_names, skiprows=0)
else:
    print(f"Error: {coord_history_path_1} not found. Please ensure the MDO run generated this file.")

if os.path.exists(coord_history_path_2):
    with open(coord_history_path_2, 'r') as f:
        file_content = f.read()
        df_coord_results_2 = pd.read_csv(io.StringIO(file_content), sep=',\s*', engine='python', names=column_names, skiprows=0)
else:
    print(f"Error: {coord_history_path_2} not found. Please ensure the MDO run generated this file.")
# Read the data using pandas, specifying the delimiter and column names
# Use io.StringIO to treat the string content as a file


# Convert 'Iteration #' and 'Objective' and 'Max. inconsistency' to numeric, coercing errors
df_coord_results_1['Iteration #'] = pd.to_numeric(df_coord_results_1['Iteration #'], errors='coerce')
df_coord_results_1['Objective'] = pd.to_numeric(df_coord_results_1['Objective'], errors='coerce')
df_coord_results_1['Max. inconsistency'] = pd.to_numeric(df_coord_results_1['Max. inconsistency'], errors='coerce')

# Drop rows where key columns might be NaN after coercion (e.g., if header was parsed as data)
df_coord_results_1.dropna(subset=['Iteration #', 'Objective', 'Max. inconsistency'], inplace=True)

# Convert 'Iteration #' and 'Objective' and 'Max. inconsistency' to numeric, coercing errors
df_coord_results_2['Iteration #'] = pd.to_numeric(df_coord_results_2['Iteration #'], errors='coerce')
df_coord_results_2['Objective'] = pd.to_numeric(df_coord_results_2['Objective'], errors='coerce')
df_coord_results_2['Max. inconsistency'] = pd.to_numeric(df_coord_results_2['Max. inconsistency'], errors='coerce')

# Drop rows where key columns might be NaN after coercion (e.g., if header was parsed as data)
df_coord_results_2.dropna(subset=['Iteration #', 'Objective', 'Max. inconsistency'], inplace=True)



if df_coord_results_1.empty:
    print(f"No valid iteration data found in {coord_history_path_1}. Please ensure the MDO run completed and generated output in the expected format.")
if df_coord_results_2.empty:
    print(f"No valid iteration data found in {coord_history_path_2}. Please ensure the MDO run completed and generated output in the expected format.")

# Plot Objective Function Convergence
fig_coord_obj_1 = px.line(df_coord_results_1, x='Iteration #', y='Objective', title=f'<b>Coordination: Objective Function Convergence</b>')
fig_coord_obj_2 = px.line(df_coord_results_2, x='Iteration #', y='Objective', title=f'<b>Coordination: Objective Function Convergence</b>')
fig_coord_obj_1.add_traces(fig_coord_obj_2.data)

# 3. Apply specific colors to each trace
# Trace 0 is the original fig_coord_obj_1
fig_coord_obj_1.data[0].name = "Serial"
fig_coord_obj_1.data[0].showlegend = True
fig_coord_obj_1.data[0].line.color = "cyan"

# Trace 1 is the added data from fig_coord_obj_2
fig_coord_obj_1.data[1].name = "Parallel"
fig_coord_obj_1.data[1].showlegend = True
fig_coord_obj_1.data[1].line.color = "orange"

fig_coord_obj_1.update_traces(mode='lines+markers')
fig_coord_obj_1.update_layout(template="plotly_dark", title_x=0.5, showlegend=True, legend=dict(
        orientation="h",      # Horizontal orientation
        yanchor="bottom",
        y=1.02,               # Moves it to the top of the plot
        xanchor="center",
        x=0.5,                # Centers it horizontally
        font=dict(size=14)    # Increase font size
    )) # Set dark background and center title
fig_coord_obj_1.update_yaxes(type="log") # Set y-axis to logarithmic scale

# fig_coord_obj_1.update_layout(
#     showlegend=True,
#     legend=dict(
#         yanchor="top",
#         y=0.99,
#         xanchor="right",
#         x=0.99
#     )
# )

fig_coord_obj_1.show()

# Plot Max Inconsistency Convergence
fig_coord_inconsistency_1 = px.line(df_coord_results_1, x='Iteration #', y='Max. inconsistency', title=f'<b>Coordination: Max Inconsistency Convergence</b>')
fig_coord_inconsistency_2 = px.line(df_coord_results_2, x='Iteration #', y='Max. inconsistency', title=f'<b>Coordination: Max Inconsistency Convergence</b>')
fig_coord_inconsistency_1.add_traces(fig_coord_inconsistency_2.data)

# Trace 0 is the original fig_coord_obj_1
fig_coord_inconsistency_1.data[0].name = "Serial"
fig_coord_inconsistency_1.data[0].showlegend = True
fig_coord_inconsistency_1.data[0].line.color = "cyan"

# Trace 1 is the added data from fig_coord_obj_2
fig_coord_inconsistency_1.data[1].name = "Parallel"
fig_coord_inconsistency_1.data[1].showlegend = True
fig_coord_inconsistency_1.data[1].line.color = "orange"

fig_coord_inconsistency_1.update_traces(mode='lines+markers')
fig_coord_inconsistency_1.update_layout(template="plotly_dark", title_x=0.5, showlegend=True, legend=dict(
        orientation="h",      # Horizontal orientation
        yanchor="bottom",
        y=1.02,               # Moves it to the top of the plot
        xanchor="center",
        x=0.5,                # Centers it horizontally
        font=dict(size=14)    # Increase font size
    ))
fig_coord_inconsistency_1.update_yaxes(type="log") # Set y-axis to logarithmic scale

# fig_coord_inconsistency_1.update_layout(
#     showlegend=True,
#     legend=dict(
#         yanchor="top",
#         y=0.99,
#         xanchor="right",
#         x=0.99
#     )
# )

fig_coord_inconsistency_1.show()


# References

<a name="ref1"></a> [1] Tosserams, S., Kokkolaras, M., Etman, L. F. P., & Rooda, J. E. (2010). A nonhierarchical formulation of analytical target cascading.

<a name="ref2"></a> [2] Talgorn, B., and Kokkolaras, M.. "Compact implementation of non-hierarchical analytical target cascading for coordinating distributed multidisciplinary design optimization problems." Structural and Multidisciplinary Optimization 56, no. 6 (2017): 1597-1602.

